In [1]:
import os, sys
pf_path = r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9"
os.environ["PATH"] = pf_path + ";" + os.environ["PATH"]
sys.path.append(pf_path)

import powerfactory as pf
app = pf.GetApplication()
if app is None:
    raise Exception("Gagal konek ke PowerFactory")


In [2]:
import powerfactory as pf

app = pf.GetApplication()
app.Show()
app.ResetCalculation()



In [4]:
PROJECT = "SS JATIM 2025"
ret = app.ActivateProject(PROJECT)
if ret != 0:
    raise Exception(f"❌ Gagal aktivasi project: {PROJECT}")


Skenario 3 Beban Maksimum

In [6]:
import math
import os
import time
from itertools import combinations
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    import powerfactory
except ImportError:
    powerfactory = None


# ============================================================
# BASIC HELPER
# ============================================================

def normalize_name(name: str) -> str:
    """
    Normalisasi nama:
    - uppercase
    - trim spasi depan belakang
    - multiple spaces menjadi single space
    """
    if name is None:
        return ""
    return " ".join(str(name).strip().upper().split())


def compact_name(name: str) -> str:
    """
    Normalisasi lebih agresif untuk matching nama:
    - uppercase
    - hapus spasi
    - hapus karakter pemisah umum
    """
    if name is None:
        return ""

    text = str(name).strip().upper()
    for ch in [" ", "-", "_", ".", "/", "\\"]:
        text = text.replace(ch, "")
    return text


def make_safe_filename(text: str) -> str:
    """
    Membuat nama file aman dari spasi dan karakter khusus sederhana.
    """
    return (
        text.replace(" ", "_")
            .replace("-", "")
            .replace("/", "_")
            .replace("\\", "_")
    )


def log(app: Any, message: str) -> None:
    try:
        app.PrintPlain(message)
    except Exception:
        print(message)


# ============================================================
# PILIH SKENARIO DAN KONDISI BEBAN
# ============================================================

CURRENT_SCENARIO = "Skenario 2"
LOAD_CONDITION = "Beban Maksimum"

# Pilihan:
# CURRENT_SCENARIO = "Skenario 1", "Skenario 2", "Skenario 3"
# LOAD_CONDITION  = "Beban Rata-Rata", "Beban Maksimum"


# ============================================================
# JUMLAH BC YANG DIBUKA
# Untuk buka 1 BC saja       : [1]
# Untuk buka 2 BC saja       : [2]
# Untuk buka 1 sampai 2 BC   : [1, 2]
# ============================================================

OPEN_COUNTS = [2]

# exact:
# BC yang dipilih dibuka, BC lain di COUP_LIST ditutup
SCENARIO_MODE = "exact"


# ============================================================
# CASE NAME
# Pastikan study case aktif di PowerFactory sesuai pilihan:
# CURRENT_SCENARIO dan LOAD_CONDITION
# ============================================================

CASE_NAME = f"{CURRENT_SCENARIO} - {LOAD_CONDITION}"


# ============================================================
# DAFTAR BUS COUPLER KANDIDAT
# ============================================================

COUP_LIST = [
    "BC Waru",
    "BC Rungkut",
    "BC Sawahan",
    "BC Gunungsari",
    "BC Tandes",
    "BC Alta",
    "BC Undaan",
    "BC Kupang",
    "BC Sambikerep",
    "BC Darmo",
]


# ============================================================
# LINE LIST UNTUK PERHITUNGAN PIpf
# PIpf hanya dihitung dari line pada daftar ini
# ============================================================

LINE_LIST = [
    "GRBRU-TNDES 1",
    "GRBRU-TNDES 2",
    "ALTAP-KRIAN 1",
    "ALTAP-KRIAN 2",
    "KRIAN-DARMO 1",
    "KRIAN-DARMO 2",
    "KRIAN-KLANG 1",
    "KRIAN-KLANG 2",
    "KRIAN-SWHAN1",
    "KRIAN-SWHAN2",
    "GRLMA - SMKRP 1A",
    "GRLMA - SMKRP 2A",
    "GRLMA-ALTAP 1",
    "GRLMA-ALTAP 2",
    "GRLMA-SGMDU 1",
    "GRLMA-SGMDU 2",
    "Grlma-T.Wlmar",
    "KLANG-BAMBE 1",
    "KLANG-BAMBE 2",
    "SWHAN-GNSRI 1",
    "SWHAN-GNSRI 2",
    "SWHAN-KPANG 1",
    "SWHAN-KPANG 2",
    "SWHAN-KRBAN 1",
    "SWHAN-KRBAN 2",
    "SWHAN-UDAAN 1",
    "SWHAN-UDAAN 2",
    "TNDES-SWHAN 1",
    "TNDES-SWHAN 2",
    "DARMO-WARU 1",
    "DARMO-WARU 2",
    "SGMDU-ALTAP 1",
    "SGMDU-ALTAP 2",
    "SMKRP - WARU A 1",
    "SMKRP - WARU A 2",
    "WARU-GNSRI 1",
    "WARU-GNSRI 2",
    "WARU-BDRAN 2",
    "WARU-NBDRN",
    "WARU-PTJTS",
    "WARU-RNKUT 1",
    "WARU-RNKUT 2",
    "RNKUT-HANIL",
    "RNKUT-SLILO 1",
    "RNKUT-SLILO 2",
    "SBSLN-RNKUT 1",
    "SBSLN-RNKUT 2",
    "BDRAN-NBRAN 1",
    "BDRAN-SDRJO 1B",
    "BDRAN-SDRJO 2B",
    "KLSARI-SUBSEL-I",
    "KLSARI-SUBSEL-II",
    "SLILO-KJRAN 1",
    "SLILO-KJRAN 2",
    "SLILO-KLSRI 1A",
    "SLILO-KLSRI 2A",
    "SLILO-NGAGL 1",
    "SLILO-NGAGL 2",
    "SLILO-WKRMO 1",
    "SLILO-WKRMO 2",
    "NGAGL-SIMPG 1",
    "NGAGL-SIMPG 2",
    "KJRAN-KDING 1A",
    "SDATI-NBDRAN 1",
    "SDATI-NBDRAN 2",
    "GLTMR-BKLAN 1",
    "GLTMR-KJRAN",
    "TANDES-Tx PERAK",
    "TANDES-UJUNG 1",
    "PERAK-INDSM",
    "UDAAN-GBONG 1",
    "UDAAN-GBONG 2",
    "Bklan-CH.Ujung",
    "Kding-CH.Ujung",
    "T.Perak-CH.Ujung2",
    "SURAMADU 4",
    "BANGKALAN - SAMPANG 1",
    "BANGKALAN - SAMPANG 2",
    "BKLAN-SRMDU 3B",
    "SAMPG-PMKSN 1",
    "SAMPG-PMKSN 2",
    "PMKSN-GULUK 1",
    "PMKSN-GULUK 2",
    "SMNEP-GULUK1",
    "SMNEP-GULUK2",
    "NWARU-WARU1",
    "NWARU-WARU 2",
    "NWARU-RNKUT 1",
    "NWARU-RNKUT 2"
  
]


# ============================================================
# DAFTAR GI / SUBSTATION YANG DIMONITOR UNTUK PIsc DAN PIv
# ============================================================

SUBSTAT_LIST = [
    "GRESIK BARU5", "KRIAN5", "GRESIK LAMA55", "KARANGPILANG5", "SAWAHAN5",
    "DARMO GRANDE5", "ALTA PRIMA5", "SEGOROMADU5", "SAMBIKEREP5", "WILMAR5",
    "BAMBE5", "GUNUNG SARI5", "PETRO KIMIA5", "WARU5", "RUNGKUT5",
    "BUDURAN5", "SURABAYA SELATAN5", "SUKOLILO5", "HANIL JAYA STEEL5",
    "JATIM TAMAN STEEL5", "KALISARI5", "WONOKROMO5", "NGAGEL5", "KENJERAN5",
    "SIDOARJO5", "NEWBUDURAN5", "SEDATI5", "SIMPANG5", "GILITIMUR5",
    "NWARU5", "TANDES5", "PERAK5", "UNDAAN5", "KUPANG5","GEMBONG5",
    "KREMBANGAN5", "UJUNG5", "INDOFOOD SUKSES MAKMUR5", "KEDINDING5",
    "BANGKALAN5", "SAMPANG5", "PAMEKASAN5", "GULUKGULUK5", "SUMENEP5",
]


# ============================================================
# BASELINE PI PER SKENARIO DAN KONDISI BEBAN
# Nilai ini digunakan sebagai pembagi PI norm.
# PIsc baseline sudah memakai:
# KRIAN5 dan NWARU5 = 50 kA
# GI lain = 40 kA
# ============================================================

BASELINE_PI = {
    "Skenario 1": {
        "Beban Rata-Rata": {
            "PIsc": 0.00,
            "PIv": 1.18,
            "PIpf": 4.27,
        },
        "Beban Maksimum": {
            "PIsc": 0.00,
            "PIv": 21.80,
            "PIpf": 9.15,
        },
    },
    "Skenario 2": {
        "Beban Rata-Rata": {
            "PIsc": 14.74,
            "PIv": 1.41,
            "PIpf": 4.60,
        },
        "Beban Maksimum": {
            "PIsc": 14.74,
            "PIv": 4.21,
            "PIpf": 6.93,
        },
    },
    "Skenario 3": {
        "Beban Rata-Rata": {
            "PIsc": 29.46,
            "PIv": 1.76,
            "PIpf": 2.87,
        },
        "Beban Maksimum": {
            "PIsc": 29.46,
            "PIv": 3.13,
            "PIpf": 5.75,
        },
    },
}


def get_selected_baseline(
    scenario_name: str,
    load_condition: str,
) -> Tuple[float, float, float]:
    """
    Mengambil baseline PI berdasarkan skenario dan kondisi beban.

    Return:
        BASELINE_PISC, BASELINE_PIV, BASELINE_PIPF
    """
    if scenario_name not in BASELINE_PI:
        raise ValueError(f"Skenario tidak ditemukan: {scenario_name}")

    if load_condition not in BASELINE_PI[scenario_name]:
        raise ValueError(f"Kondisi beban tidak ditemukan: {load_condition}")

    baseline = BASELINE_PI[scenario_name][load_condition]

    return (
        baseline["PIsc"],
        baseline["PIv"],
        baseline["PIpf"],
    )


BASELINE_PISC, BASELINE_PIV, BASELINE_PIPF = get_selected_baseline(
    CURRENT_SCENARIO,
    LOAD_CONDITION,
)


# ============================================================
# BOBOT AHP
# Urutan kriteria:
# PIsc -> PIv -> PIpf
# PIsc dominan
#
# PI Total =
# 0.637 * PIsc_norm +
# 0.258 * PIv_norm  +
# 0.105 * PIpf_norm
# ============================================================

WEIGHT_SC = 0.637
WEIGHT_V = 0.258
WEIGHT_PF = 0.105


# ============================================================
# PARAMETER PIsc
# KRIAN5 dan NWARU5 memakai limit 50 kA.
# GI lainnya memakai limit 40 kA.
# ============================================================

DEFAULT_SCC_LIMIT_KA = 40.0

SPECIAL_SCC_LIMIT_KA = {
    "KRIAN5": 50.0,
    "NWARU5": 50.0,
}


def get_scc_limit_ka(substation_name: str) -> float:
    """
    Mengambil batas arus hubung singkat berdasarkan nama substation.

    KRIAN5 dan NWARU5 = 50 kA.
    Selain itu = 40 kA.
    """
    name = normalize_name(substation_name)
    return SPECIAL_SCC_LIMIT_KA.get(name, DEFAULT_SCC_LIMIT_KA)


# ============================================================
# PARAMETER PIv
# PIv_i = 0.5 * (|Vi - 1.0| / DeltaVlim)^2
# Jika Vi < 1.0 pu, DeltaVlim = 0.05
# Jika Vi > 1.0 pu, DeltaVlim = 0.10
# ============================================================

V_RATED_PU = 1.0
DV_LIMIT_UNDER_PU = 0.05
DV_LIMIT_OVER_PU = 0.10


# ============================================================
# PARAMETER NORMALISASI
# ============================================================

EPSILON = 1e-12
ZERO_BASELINE_PENALTY = 1_000_000.0


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_DIR = r"D:\Robbyo\S2 UGM\2026\Tesis\Hasil Uji\Final Revisi"

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    f"{make_safe_filename(CASE_NAME)}_28_Mei_2_CB.xlsx"
)

LOG_EACH_SCENARIO = True
SAVE_DETAIL_EACH_BC = True


# ============================================================
# POWERFACTORY APP
# ============================================================

def get_app() -> Any:
    if powerfactory is None:
        raise RuntimeError("powerfactory module not available. Run this script inside DIgSILENT PowerFactory.")

    app = powerfactory.GetApplication()
    if not app:
        raise RuntimeError("Run this script inside DIgSILENT PowerFactory.")

    log(app, "===================================================")
    log(app, f"START: {CASE_NAME}")
    log(app, f"CURRENT_SCENARIO = {CURRENT_SCENARIO}")
    log(app, f"LOAD_CONDITION  = {LOAD_CONDITION}")
    log(app, f"OPEN_COUNTS     = {OPEN_COUNTS}")
    log(app, f"Baseline PIsc   = {BASELINE_PISC}")
    log(app, f"Baseline PIv    = {BASELINE_PIV}")
    log(app, f"Baseline PIpf   = {BASELINE_PIPF}")
    log(app, "PIpf source: LINE_LIST only")
    log(app, "SCC Limit: KRIAN5 & NWARU5 = 50 kA, lainnya = 40 kA")
    log(app, "PI Total = 0.637*PIsc_norm + 0.258*PIv_norm + 0.105*PIpf_norm")
    log(app, "===================================================")

    return app


def safe_get_attribute(obj: Any, attr: str, default: Any = None) -> Any:
    try:
        value = obj.GetAttribute(attr)
        return default if value is None else value
    except Exception:
        return default


def safe_loc_name(obj: Any, default: str = "") -> str:
    if obj is None:
        return default

    try:
        return obj.loc_name.strip()
    except Exception:
        return default


def as_float(value: Any, default: Optional[float] = None) -> Optional[float]:
    try:
        number = float(value)
    except Exception:
        return default

    if not math.isfinite(number):
        return default

    return number


def get_obj_key(obj: Any) -> Optional[str]:
    if obj is None:
        return None

    try:
        return obj.GetFullName()
    except Exception:
        pass

    try:
        return f"{obj.GetClassName()}::{obj.loc_name.strip()}"
    except Exception:
        return str(obj)


def get_parent_substation_name(obj: Any, max_depth: int = 20) -> Optional[str]:
    current = obj

    for _ in range(max_depth):
        if current is None:
            return None

        try:
            if current.GetClassName() == "ElmSubstat":
                return current.loc_name.strip()

            current = current.GetParent()
        except Exception:
            return None

    return None


def resolve_terminal(side_obj: Any) -> Any:
    if side_obj is None:
        return None

    try:
        if side_obj.GetClassName() == "ElmTerm":
            return side_obj
    except Exception:
        pass

    try:
        terminal = side_obj.cterm
        if terminal is not None:
            return terminal
    except Exception:
        pass

    return None


# ============================================================
# POWERFACTORY COMMAND
# ============================================================

def run_loadflow(app: Any) -> bool:
    lf = app.GetFromStudyCase("ComLdf")
    if lf is None:
        raise RuntimeError("ComLdf not found in active study case.")

    lf.iopt_net = 0
    lf.iopt_o4 = 0

    return lf.Execute() == 0


def prepare_short_circuit(app: Any) -> Any:
    sc = app.GetFromStudyCase("ComShc")
    if sc is None:
        raise RuntimeError("ComShc not found in active study case.")

    sc.iopt_mde = 1
    sc.Rf = 0
    sc.Xf = 0
    sc.iopt_allbus = 0
    sc.iopt_shc = "3psc"

    return sc


# ============================================================
# CACHE OBJECTS
# ============================================================

def cache_couplers(app: Any) -> Tuple[Dict[str, Dict[str, Any]], List[str]]:
    couplers: Dict[str, Dict[str, Any]] = {}

    target_map_normal = {
        normalize_name(name): name
        for name in COUP_LIST
    }

    target_map_compact = {
        compact_name(name): name
        for name in COUP_LIST
    }

    found_targets = set()

    for coupler in app.GetCalcRelevantObjects("*.ElmCoup"):
        pf_name = safe_loc_name(coupler)
        pf_key_normal = normalize_name(pf_name)
        pf_key_compact = compact_name(pf_name)

        target_name = None

        if pf_key_normal in target_map_normal:
            target_name = target_map_normal[pf_key_normal]
        elif pf_key_compact in target_map_compact:
            target_name = target_map_compact[pf_key_compact]

        if target_name:
            couplers[target_name] = {
                "obj": coupler,
                "orig_open": bool(coupler.IsOpen()),
                "pf_name": pf_name,
            }
            found_targets.add(target_name)

    ordered_names = [name for name in COUP_LIST if name in couplers]

    missing_couplers = [name for name in COUP_LIST if name not in found_targets]

    log(app, f"Found monitored couplers: {len(ordered_names)}")
    for name in ordered_names:
        log(app, f" - {name} | PF name: {couplers[name].get('pf_name')}")

    if missing_couplers:
        log(app, "Warning: beberapa BC di COUP_LIST tidak ditemukan:")
        for name in missing_couplers:
            log(app, f" - {name}")

    return couplers, ordered_names


def cache_buses(app: Any) -> Tuple[List[Dict[str, Any]], set, set]:
    substations = {
        safe_loc_name(substation): substation
        for substation in app.GetCalcRelevantObjects("*.ElmSubstat")
    }

    # Tambahan lookup dengan normalisasi agar nama GI lebih aman
    substations_norm = {
        normalize_name(name): substation
        for name, substation in substations.items()
    }

    buses: List[Dict[str, Any]] = []
    monitored_substation_names = set()
    monitored_bus_keys = set()

    for sub_name in SUBSTAT_LIST:
        substation = substations.get(sub_name)
        if substation is None:
            substation = substations_norm.get(normalize_name(sub_name))

        if substation is None:
            log(app, f"Warning: Substation not found: {sub_name}")
            continue

        actual_sub_name = safe_loc_name(substation) or sub_name
        monitored_substation_names.add(actual_sub_name)

        all_terms = list(substation.GetContents("*.ElmTerm", 1) or [])

        for bus in all_terms:
            usage = safe_get_attribute(bus, "iUsage")

            try:
                if int(usage) != 0:
                    continue
            except Exception:
                continue

            key = get_obj_key(bus)

            if not key or key in monitored_bus_keys:
                continue

            monitored_bus_keys.add(key)

            buses.append({
                "substation": actual_sub_name,
                "bus_name": safe_loc_name(bus),
                "obj": bus,
                "key": key,
            })

    log(app, f"Total monitored buses from SUBSTAT_LIST = {len(buses)}")

    return buses, monitored_bus_keys, monitored_substation_names


def get_line_terminals(line_obj: Any) -> Tuple[Any, Any]:
    cub1 = None
    cub2 = None

    try:
        cub1 = line_obj.bus1
    except Exception:
        pass

    try:
        cub2 = line_obj.bus2
    except Exception:
        pass

    if cub1 is None:
        cub1 = safe_get_attribute(line_obj, "bus1")

    if cub2 is None:
        cub2 = safe_get_attribute(line_obj, "bus2")

    term1 = resolve_terminal(cub1)
    term2 = resolve_terminal(cub2)

    return term1, term2


def cache_lines(
    app: Any,
    monitored_bus_keys: set = None,
    monitored_substation_names: set = None,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Cache line hanya berdasarkan LINE_LIST.
    monitored_bus_keys dan monitored_substation_names tetap diterima supaya kompatibel,
    tetapi tidak digunakan untuk filter PIpf.
    """

    lines: List[Dict[str, Any]] = []
    missing_rows: List[Dict[str, Any]] = []
    seen_line_keys = set()

    target_line_map_normal = {
        normalize_name(line_name): line_name
        for line_name in LINE_LIST
    }

    target_line_map_compact = {
        compact_name(line_name): line_name
        for line_name in LINE_LIST
    }

    found_target_names = set()

    for line in app.GetCalcRelevantObjects("*.ElmLne"):
        line_name_pf = safe_loc_name(line)
        line_key_normal = normalize_name(line_name_pf)
        line_key_compact = compact_name(line_name_pf)

        target_name = None

        if line_key_normal in target_line_map_normal:
            target_name = target_line_map_normal[line_key_normal]
        elif line_key_compact in target_line_map_compact:
            target_name = target_line_map_compact[line_key_compact]

        if target_name is None:
            continue

        outserv = safe_get_attribute(line, "outserv", 0)

        try:
            if int(outserv) == 1:
                missing_rows.append({
                    "Target Line Name": target_name,
                    "PowerFactory Name": line_name_pf,
                    "Status": "FOUND_BUT_OUT_OF_SERVICE",
                })
                found_target_names.add(target_name)
                continue
        except Exception:
            pass

        line_obj_key = get_obj_key(line)

        if not line_obj_key or line_obj_key in seen_line_keys:
            continue

        seen_line_keys.add(line_obj_key)
        found_target_names.add(target_name)

        term1, term2 = get_line_terminals(line)

        term1_sub = get_parent_substation_name(term1) or ""
        term2_sub = get_parent_substation_name(term2) or ""

        lines.append({
            "line_name": line_name_pf,
            "line_name_target": target_name,
            "selected_side": "max_current_side",
            "obj": line,
            "bus_name": f"{safe_loc_name(term1)} / {safe_loc_name(term2)}",
            "substation": f"{term1_sub} / {term2_sub}",
        })

    for original_name in LINE_LIST:
        if original_name not in found_target_names:
            missing_rows.append({
                "Target Line Name": original_name,
                "PowerFactory Name": None,
                "Status": "NOT_FOUND",
            })

    log(app, f"Total monitored lines from LINE_LIST = {len(lines)}")
    log(app, f"Total missing/out-service lines from LINE_LIST = {len(missing_rows)}")

    if missing_rows:
        log(app, "Warning: beberapa line di LINE_LIST tidak ditemukan atau out of service:")
        for row in missing_rows:
            log(app, f" - {row['Target Line Name']} | {row['Status']}")

    return lines, missing_rows


# ============================================================
# COUPLER OPERATION
# ============================================================

def restore_couplers(couplers: Dict[str, Dict[str, Any]]) -> None:
    for info in couplers.values():
        obj = info["obj"]
        orig_open = info["orig_open"]

        try:
            current_open = bool(obj.IsOpen())
        except Exception:
            continue

        if current_open != orig_open:
            try:
                if orig_open:
                    obj.Open()
                else:
                    obj.Close()
            except Exception:
                continue


def apply_open_scenario(
    couplers: Dict[str, Dict[str, Any]],
    open_names: Sequence[str],
    scenario_mode: str = SCENARIO_MODE,
) -> None:

    open_set = set(open_names)

    if scenario_mode not in {"exact", "relative"}:
        raise ValueError(f"Unsupported SCENARIO_MODE: {scenario_mode}")

    for name, info in couplers.items():
        obj = info["obj"]

        if name in open_set:
            obj.Open()
        elif scenario_mode == "exact":
            obj.Close()


# ============================================================
# PERFORMANCE INDEX - PIpf
# ============================================================

def calculate_pipf_total(
    lines: Sequence[Dict[str, Any]],
    save_details: bool = False,
) -> Tuple[float, List[Dict[str, Any]]]:

    total_pipf = 0.0
    details: List[Dict[str, Any]] = []

    for item in lines:
        line_obj = item["obj"]

        loading = as_float(safe_get_attribute(line_obj, "m:loading", 0.0), 0.0)

        i_bus1 = as_float(safe_get_attribute(line_obj, "m:I1:bus1", 0.0), 0.0)
        i_bus2 = as_float(safe_get_attribute(line_obj, "m:I1:bus2", 0.0), 0.0)

        i_bus1 = abs(i_bus1 or 0.0)
        i_bus2 = abs(i_bus2 or 0.0)

        i_actual = max(i_bus1, i_bus2)

        i_limit = as_float(safe_get_attribute(line_obj, "c:curnom", 0.0), 0.0)
        i_limit = i_limit or 0.0

        if i_limit <= 0.0:
            pipf_i = 0.0
        else:
            pipf_i = 0.5 * ((i_actual / i_limit) ** 2)

        total_pipf += pipf_i

        if save_details:
            details.append({
                "Substation": item["substation"],
                "Bus": item["bus_name"],
                "Line": item["line_name"],
                "Target Line Name": item.get("line_name_target"),
                "Selected Side": "Max(bus1,bus2)",
                "Loading (%)": round(loading, 6),
                "I bus1 (kA)": round(i_bus1, 6),
                "I bus2 (kA)": round(i_bus2, 6),
                "I actual max (kA)": round(i_actual, 6),
                "I limit (kA)": round(i_limit, 6),
                "PIpf_i = 0.5*(Iactual/Ilimit)^2": round(float(pipf_i), 8),
            })

    return total_pipf, details


# ============================================================
# PERFORMANCE INDEX - PIsc
# ============================================================

def calculate_pisc_total(
    sc: Any,
    buses: Sequence[Dict[str, Any]],
    save_details: bool = False,
) -> Tuple[Optional[float], List[Dict[str, Any]], str]:

    total_pisc = 0.0
    details: List[Dict[str, Any]] = []

    for item in buses:
        substation_name = item["substation"]
        bus = item["obj"]
        sc.shcobj = bus

        if sc.Execute() != 0:
            return None, details, f"SC_EXECUTE_FAILED: {item['substation']} / {item['bus_name']}"

        isc = as_float(safe_get_attribute(bus, "m:Ikss", None), None)

        if isc is None:
            return None, details, f"SC_RESULT_MISSING: {item['substation']} / {item['bus_name']}"

        scc_limit_ka = get_scc_limit_ka(substation_name)

        if isc > scc_limit_ka:
            pisc_i = (isc / scc_limit_ka) ** 2
            violation = "YES"
        else:
            pisc_i = 0.0
            violation = "NO"

        total_pisc += pisc_i

        if save_details:
            details.append({
                "Substation": item["substation"],
                "Bus": item["bus_name"],
                "Isc / Ikss (kA)": round(isc, 6),
                "SCC Limit Used (kA)": round(scc_limit_ka, 6),
                "PIsc_i = (Isc/SCC Limit)^2 if Isc>Limit": round(float(pisc_i), 8),
                "Violation": violation,
            })

    return total_pisc, details, "OK"


# ============================================================
# PERFORMANCE INDEX - PIv
# ============================================================

def calculate_piv_total(
    buses: Sequence[Dict[str, Any]],
    save_details: bool = False,
) -> Tuple[Optional[float], List[Dict[str, Any]], str]:

    total_piv = 0.0
    details: List[Dict[str, Any]] = []

    for item in buses:
        sub_name = item["substation"]
        bus_name = item["bus_name"]
        bus_obj = item["obj"]

        vpu = as_float(safe_get_attribute(bus_obj, "m:u", None), None)
        vkv = as_float(safe_get_attribute(bus_obj, "m:Ul", None), None)

        if vpu is None:
            return None, details, f"VOLTAGE_RESULT_MISSING: {sub_name} / {bus_name}"

        deviation = abs(vpu - V_RATED_PU)

        if vpu < V_RATED_PU:
            dv_limit = DV_LIMIT_UNDER_PU
        elif vpu > V_RATED_PU:
            dv_limit = DV_LIMIT_OVER_PU
        else:
            dv_limit = DV_LIMIT_UNDER_PU

        piv_i = 0.5 * ((deviation / dv_limit) ** 2)

        total_piv += piv_i

        if vpu < (V_RATED_PU - DV_LIMIT_UNDER_PU):
            status = "LOW"
        elif vpu > (V_RATED_PU + DV_LIMIT_OVER_PU):
            status = "HIGH"
        else:
            status = "NORMAL"

        if save_details:
            details.append({
                "Substation": sub_name,
                "Bus": bus_name,
                "Vpu": round(vpu, 6),
                "VkV": round(vkv, 3) if vkv is not None else None,
                "Vrated_pu": V_RATED_PU,
                "DeltaVlim_pu": dv_limit,
                "Deviation": round(deviation, 8),
                "PIv_i = 0.5*(Deviation/DeltaVlim)^2": round(float(piv_i), 8),
                "Status": status,
            })

    return total_piv, details, "OK"


# ============================================================
# NORMALISASI DAN PI TOTAL
# ============================================================

def normalize_by_baseline(value: Optional[float], baseline: float) -> Optional[float]:
    if value is None:
        return None

    if abs(baseline) <= EPSILON:
        if abs(value) <= EPSILON:
            return 1.0
        return ZERO_BASELINE_PENALTY

    return value / baseline


def calculate_pi_total(
    pisc_total: float,
    piv_total: float,
    pipf_total: float,
) -> Tuple[
    Optional[float],
    Optional[float],
    Optional[float],
    Optional[float],
    Optional[float],
    Optional[float],
    Optional[float],
]:

    pisc_norm = normalize_by_baseline(pisc_total, BASELINE_PISC)
    piv_norm = normalize_by_baseline(piv_total, BASELINE_PIV)
    pipf_norm = normalize_by_baseline(pipf_total, BASELINE_PIPF)

    if pisc_norm is None or piv_norm is None or pipf_norm is None:
        return None, pisc_norm, piv_norm, pipf_norm, None, None, None

    pisc_contribution = WEIGHT_SC * pisc_norm
    piv_contribution = WEIGHT_V * piv_norm
    pipf_contribution = WEIGHT_PF * pipf_norm

    pi_total = pisc_contribution + piv_contribution + pipf_contribution

    return (
        pi_total,
        pisc_norm,
        piv_norm,
        pipf_norm,
        pisc_contribution,
        piv_contribution,
        pipf_contribution,
    )


# ============================================================
# EVALUASI SETIAP PEMBUKAAN BC
# ============================================================

def evaluate_open_set(
    app: Any,
    couplers: Dict[str, Dict[str, Any]],
    lines: Sequence[Dict[str, Any]],
    buses: Sequence[Dict[str, Any]],
    open_names: Sequence[str],
    save_details: bool = False,
) -> Dict[str, Any]:

    restore_couplers(couplers)

    try:
        apply_open_scenario(couplers, open_names, SCENARIO_MODE)

        if not run_loadflow(app):
            return {
                "status": "LOADFLOW_FAILED",
                "pipf_total": None,
                "piv_total": None,
                "pisc_total": None,
                "line_details": [],
                "v_details": [],
                "sc_details": [],
            }

        pipf_total, line_details = calculate_pipf_total(
            lines,
            save_details=save_details,
        )

        piv_total, v_details, piv_status = calculate_piv_total(
            buses,
            save_details=save_details,
        )

        if piv_total is None:
            return {
                "status": piv_status,
                "pipf_total": pipf_total,
                "piv_total": None,
                "pisc_total": None,
                "line_details": line_details,
                "v_details": v_details,
                "sc_details": [],
            }

        sc = prepare_short_circuit(app)

        pisc_total, sc_details, pisc_status = calculate_pisc_total(
            sc,
            buses,
            save_details=save_details,
        )

        if pisc_total is None:
            return {
                "status": pisc_status,
                "pipf_total": pipf_total,
                "piv_total": piv_total,
                "pisc_total": None,
                "line_details": line_details,
                "v_details": v_details,
                "sc_details": sc_details,
            }

        (
            pi_total,
            pisc_norm,
            piv_norm,
            pipf_norm,
            pisc_contribution,
            piv_contribution,
            pipf_contribution,
        ) = calculate_pi_total(
            pisc_total=pisc_total,
            piv_total=piv_total,
            pipf_total=pipf_total,
        )

        return {
            "status": "OK",
            "pipf_total": pipf_total,
            "piv_total": piv_total,
            "pisc_total": pisc_total,

            "pisc_norm": pisc_norm,
            "piv_norm": piv_norm,
            "pipf_norm": pipf_norm,

            "pisc_contribution": pisc_contribution,
            "piv_contribution": piv_contribution,
            "pipf_contribution": pipf_contribution,
            "pi_total": pi_total,

            "line_details": line_details,
            "v_details": v_details,
            "sc_details": sc_details,
        }

    except Exception as exc:
        return {
            "status": f"ERROR: {exc}",
            "pipf_total": None,
            "piv_total": None,
            "pisc_total": None,
            "line_details": [],
            "v_details": [],
            "sc_details": [],
        }

    finally:
        restore_couplers(couplers)


def build_ranking(df_all: pd.DataFrame, score_column: str) -> pd.DataFrame:
    if df_all.empty or score_column not in df_all.columns:
        return pd.DataFrame()

    df_valid = df_all[
        (df_all["Status"] == "OK") &
        (df_all[score_column].notna())
    ]

    if df_valid.empty:
        return pd.DataFrame()

    return df_valid.sort_values(by=score_column, ascending=True).reset_index(drop=True)


# ============================================================
# MAIN PROGRAM
# ============================================================

def run_optimization_pi_total() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    app = get_app()

    t0 = time.perf_counter()

    couplers, coupler_names = cache_couplers(app)
    buses, monitored_bus_keys, monitored_substation_names = cache_buses(app)
    lines, missing_line_rows = cache_lines(
        app,
        monitored_bus_keys,
        monitored_substation_names,
    )

    if not coupler_names:
        raise RuntimeError("No couplers found from COUP_LIST.")

    if not buses:
        raise RuntimeError("No buses found from SUBSTAT_LIST.")

    if not lines:
        raise RuntimeError("No lines found from LINE_LIST.")

    scenario_rows: List[Dict[str, Any]] = []
    detail_line_rows: List[Dict[str, Any]] = []
    detail_sc_rows: List[Dict[str, Any]] = []
    detail_v_rows: List[Dict[str, Any]] = []

    best_pi_total = float("inf")
    best_open_names: Optional[List[str]] = None
    best_details = {
        "line": [],
        "sc": [],
        "v": [],
    }

    total_scenarios = sum(
        math.comb(len(coupler_names), n)
        for n in OPEN_COUNTS
        if n <= len(coupler_names)
    )

    log(app, "===================================================")
    log(app, f"Case             : {CASE_NAME}")
    log(app, f"Baseline PIsc    : {BASELINE_PISC}")
    log(app, f"Baseline PIv     : {BASELINE_PIV}")
    log(app, f"Baseline PIpf    : {BASELINE_PIPF}")
    log(app, f"Weight PIsc      : {WEIGHT_SC}")
    log(app, f"Weight PIv       : {WEIGHT_V}")
    log(app, f"Weight PIpf      : {WEIGHT_PF}")
    log(app, f"SCC Default Limit: {DEFAULT_SCC_LIMIT_KA} kA")
    log(app, f"SCC Special Limit: KRIAN5={SPECIAL_SCC_LIMIT_KA['KRIAN5']} kA, NWARU5={SPECIAL_SCC_LIMIT_KA['NWARU5']} kA")
    log(app, f"Total line PIpf  : {len(lines)}")
    log(app, f"Total scenario   : {total_scenarios}")
    log(app, "===================================================")

    scenario_no = 1

    try:
        for n_open in OPEN_COUNTS:
            if n_open > len(coupler_names):
                continue

            log(app, f"===== START EVALUATION: OPEN {n_open} BC =====")

            for combo in combinations(coupler_names, n_open):
                open_names = list(combo)
                scenario_label = ", ".join(open_names)

                if LOG_EACH_SCENARIO:
                    log(app, f"[{scenario_no}] OPEN BC: {scenario_label}")

                result = evaluate_open_set(
                    app=app,
                    couplers=couplers,
                    lines=lines,
                    buses=buses,
                    open_names=open_names,
                    save_details=SAVE_DETAIL_EACH_BC,
                )

                status = result.get("status")

                pisc_total = result.get("pisc_total")
                piv_total = result.get("piv_total")
                pipf_total = result.get("pipf_total")

                pisc_norm = result.get("pisc_norm")
                piv_norm = result.get("piv_norm")
                pipf_norm = result.get("pipf_norm")

                pisc_contribution = result.get("pisc_contribution")
                piv_contribution = result.get("piv_contribution")
                pipf_contribution = result.get("pipf_contribution")
                pi_total = result.get("pi_total")

                line_details = result.get("line_details", [])
                sc_details = result.get("sc_details", [])
                v_details = result.get("v_details", [])

                scenario_rows.append({
                    "Case": CASE_NAME,
                    "Current Scenario": CURRENT_SCENARIO,
                    "Load Condition": LOAD_CONDITION,
                    "No": scenario_no,
                    "N Open": n_open,
                    "Opened BC": scenario_label,

                    "PIsc Total": round(pisc_total, 8) if pisc_total is not None else None,
                    "PIsc Baseline": BASELINE_PISC,
                    f"PIsc Norm = PIsc Total / {BASELINE_PISC}": round(pisc_norm, 8) if pisc_norm is not None else None,
                    "Weight PIsc": WEIGHT_SC,
                    f"PIsc Contribution = {WEIGHT_SC}*PIsc Norm": round(pisc_contribution, 8) if pisc_contribution is not None else None,

                    "PIv Total": round(piv_total, 8) if piv_total is not None else None,
                    "PIv Baseline": BASELINE_PIV,
                    f"PIv Norm = PIv Total / {BASELINE_PIV}": round(piv_norm, 8) if piv_norm is not None else None,
                    "Weight PIv": WEIGHT_V,
                    f"PIv Contribution = {WEIGHT_V}*PIv Norm": round(piv_contribution, 8) if piv_contribution is not None else None,

                    "PIpf Total": round(pipf_total, 8) if pipf_total is not None else None,
                    "PIpf Baseline": BASELINE_PIPF,
                    f"PIpf Norm = PIpf Total / {BASELINE_PIPF}": round(pipf_norm, 8) if pipf_norm is not None else None,
                    "Weight PIpf": WEIGHT_PF,
                    f"PIpf Contribution = {WEIGHT_PF}*PIpf Norm": round(pipf_contribution, 8) if pipf_contribution is not None else None,

                    "PI Total": round(pi_total, 8) if pi_total is not None else None,
                    "Status": status,
                })

                for row in sc_details:
                    detail_sc_rows.append({
                        "Case": CASE_NAME,
                        "No": scenario_no,
                        "Opened BC": scenario_label,
                        "PIsc Total": round(pisc_total, 8) if pisc_total is not None else None,
                        "PIsc Baseline": BASELINE_PISC,
                        "PIsc Norm": round(pisc_norm, 8) if pisc_norm is not None else None,
                        "PIsc Contribution": round(pisc_contribution, 8) if pisc_contribution is not None else None,
                        **row,
                    })

                for row in v_details:
                    detail_v_rows.append({
                        "Case": CASE_NAME,
                        "No": scenario_no,
                        "Opened BC": scenario_label,
                        "PIv Total": round(piv_total, 8) if piv_total is not None else None,
                        "PIv Baseline": BASELINE_PIV,
                        "PIv Norm": round(piv_norm, 8) if piv_norm is not None else None,
                        "PIv Contribution": round(piv_contribution, 8) if piv_contribution is not None else None,
                        **row,
                    })

                for row in line_details:
                    detail_line_rows.append({
                        "Case": CASE_NAME,
                        "No": scenario_no,
                        "Opened BC": scenario_label,
                        "PIpf Total": round(pipf_total, 8) if pipf_total is not None else None,
                        "PIpf Baseline": BASELINE_PIPF,
                        "PIpf Norm": round(pipf_norm, 8) if pipf_norm is not None else None,
                        "PIpf Contribution": round(pipf_contribution, 8) if pipf_contribution is not None else None,
                        **row,
                    })

                log(app, "---------------------------------------------------")
                log(app, f"OPEN BC             : {scenario_label}")
                log(app, f"Status              : {status}")

                if status == "OK":
                    log(app, f"PIsc Total          : {pisc_total:.8f}")
                    log(app, f"PIsc Norm           : {pisc_norm:.8f}")
                    log(app, f"PIsc Contribution   : {pisc_contribution:.8f}")

                    log(app, f"PIv Total           : {piv_total:.8f}")
                    log(app, f"PIv Norm            : {piv_norm:.8f}")
                    log(app, f"PIv Contribution    : {piv_contribution:.8f}")

                    log(app, f"PIpf Total          : {pipf_total:.8f}")
                    log(app, f"PIpf Norm           : {pipf_norm:.8f}")
                    log(app, f"PIpf Contribution   : {pipf_contribution:.8f}")

                    log(app, f"PI Total            : {pi_total:.8f}")

                log(app, "---------------------------------------------------")

                if status == "OK" and pi_total is not None and pi_total < best_pi_total:
                    best_pi_total = pi_total
                    best_open_names = open_names

                    best_details["line"] = list(line_details)
                    best_details["sc"] = list(sc_details)
                    best_details["v"] = list(v_details)

                    log(app, f"NEW BEST PI TOTAL = {best_pi_total:.8f} | OPEN: {scenario_label}")

                scenario_no += 1

    finally:
        restore_couplers(couplers)

    elapsed = time.perf_counter() - t0

    df_all = pd.DataFrame(scenario_rows)
    df_rank = build_ranking(df_all, "PI Total")

    best_label = ", ".join(best_open_names) if best_open_names else None

    df_best = pd.DataFrame([{
        "Case": CASE_NAME,
        "Current Scenario": CURRENT_SCENARIO,
        "Load Condition": LOAD_CONDITION,
        "Scenario Mode": SCENARIO_MODE,
        "Open Counts": str(OPEN_COUNTS),

        "Best Opened BC": best_label,
        "Best N Open": len(best_open_names) if best_open_names else None,
        "Best PI Total": round(best_pi_total, 8) if best_open_names else None,

        "Baseline PIsc": BASELINE_PISC,
        "Baseline PIv": BASELINE_PIV,
        "Baseline PIpf": BASELINE_PIPF,

        "Weight PIsc": WEIGHT_SC,
        "Weight PIv": WEIGHT_V,
        "Weight PIpf": WEIGHT_PF,

        "Default SCC Limit (kA)": DEFAULT_SCC_LIMIT_KA,
        "Special SCC Limit": "KRIAN5=50 kA; NWARU5=50 kA",

        "Total PIpf Lines Found": len(lines),
        "Total Missing/Out-service Lines": len(missing_line_rows),

        "Execution Time (s)": round(elapsed, 2),
    }])

    df_detail_sc = pd.DataFrame(detail_sc_rows)
    df_detail_v = pd.DataFrame(detail_v_rows)
    df_detail_line = pd.DataFrame(detail_line_rows)
    df_missing_line = pd.DataFrame(missing_line_rows)

    df_best_sc = pd.DataFrame([
        {
            "Case": CASE_NAME,
            "Best Opened BC": best_label,
            **row,
        }
        for row in best_details["sc"]
    ])

    df_best_v = pd.DataFrame([
        {
            "Case": CASE_NAME,
            "Best Opened BC": best_label,
            **row,
        }
        for row in best_details["v"]
    ])

    df_best_line = pd.DataFrame([
        {
            "Case": CASE_NAME,
            "Best Opened BC": best_label,
            **row,
        }
        for row in best_details["line"]
    ])

    with pd.ExcelWriter(OUTPUT_FILE) as writer:
        df_all.to_excel(writer, sheet_name="Buka_BC_Summary", index=False)
        df_rank.to_excel(writer, sheet_name="Ranking_PI_Total", index=False)
        df_best.to_excel(writer, sheet_name="Best_Result", index=False)

        if not df_detail_sc.empty:
            df_detail_sc.to_excel(writer, sheet_name="Detail_PIsc", index=False)

        if not df_detail_v.empty:
            df_detail_v.to_excel(writer, sheet_name="Detail_PIv", index=False)

        if not df_detail_line.empty:
            df_detail_line.to_excel(writer, sheet_name="Detail_PIpf_LINE_LIST", index=False)

        if not df_missing_line.empty:
            df_missing_line.to_excel(writer, sheet_name="Missing_LINE_LIST", index=False)

        if not df_best_sc.empty:
            df_best_sc.to_excel(writer, sheet_name="Best_Detail_PIsc", index=False)

        if not df_best_v.empty:
            df_best_v.to_excel(writer, sheet_name="Best_Detail_PIv", index=False)

        if not df_best_line.empty:
            df_best_line.to_excel(writer, sheet_name="Best_Detail_PIpf", index=False)

    log(app, "===================================================")
    log(app, "FINISHED")
    log(app, f"Saved Excel : {OUTPUT_FILE}")
    log(app, f"Execution   : {elapsed:.2f} s")

    if best_open_names:
        log(app, f"Best Opened BC : {best_label}")
        log(app, f"Best PI Total  : {best_pi_total:.8f}")
    else:
        log(app, "Best result not found.")

    log(app, "===================================================")


if __name__ == "__main__":
    run_optimization_pi_total()

Atur posisi DS GI Sambikerep sambil buka kopel dan ambil nilai PI total

In [6]:
import math
import os
import time
from itertools import combinations, product
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    import powerfactory
except ImportError:
    powerfactory = None


# ============================================================
# BASIC HELPER
# ============================================================

def normalize_name(name: str) -> str:
    """
    Normalisasi nama:
    - uppercase
    - trim spasi depan belakang
    - multiple spaces menjadi single space
    """
    if name is None:
        return ""
    return " ".join(str(name).strip().upper().split())


def compact_name(name: str) -> str:
    """
    Normalisasi agresif untuk matching nama:
    - uppercase
    - hapus spasi dan karakter pemisah umum
    """
    if name is None:
        return ""

    text = str(name).strip().upper()
    for ch in [" ", "-", "_", ".", "/", "\\"]:
        text = text.replace(ch, "")
    return text


def make_safe_filename(text: str) -> str:
    return (
        text.replace(" ", "_")
            .replace("-", "")
            .replace("/", "_")
            .replace("\\", "_")
    )


def log(app: Any, message: str) -> None:
    try:
        app.PrintPlain(message)
    except Exception:
        print(message)


def safe_round(value: Any, ndigits: int = 8) -> Optional[float]:
    if value is None:
        return None
    try:
        number = float(value)
    except Exception:
        return None
    if not math.isfinite(number):
        return None
    return round(number, ndigits)


def as_float(value: Any, default: Optional[float] = None) -> Optional[float]:
    try:
        number = float(value)
    except Exception:
        return default

    if not math.isfinite(number):
        return default

    return number


# ============================================================
# PILIH SKENARIO DAN KONDISI BEBAN
# Pastikan study case aktif di PowerFactory sesuai pilihan ini.
# ============================================================

CURRENT_SCENARIO = "Skenario 2"
LOAD_CONDITION = "Beban Rata-Rata"

# Pilihan:
# CURRENT_SCENARIO = "Skenario 1", "Skenario 2", "Skenario 3"
# LOAD_CONDITION  = "Beban Rata-Rata", "Beban Maksimum"


# ============================================================
# JUMLAH PMT/BC KOPEL YANG DIBUKA SETELAH KONFIGURASI DS BUS
# ============================================================

# Buka 1 BC saja     : [1]
# Buka 2 BC saja     : [2]
# Buka 1 sampai 2 BC : [1, 2]
OPEN_COUNTS = [1]

# exact:
# BC yang dipilih dibuka, BC lain dalam COUP_LIST ditutup.
SCENARIO_MODE = "exact"

CASE_NAME = f"{CURRENT_SCENARIO} - {LOAD_CONDITION}"


# ============================================================
# KONFIGURASI DS BUS
# 1 = Bus A -> DSA close, DSB open
# 0 = Bus B -> DSA open,  DSB close
# ============================================================

BAY_LIST = [
    "Waru2",
    "Waru1",
    "TD2",
    "TD1",
    "GRLAMA1",
    "GRLAMA2",
]

MIN_BAY_PER_BUS = 2
REJECT_PI_TOTAL = 9999.0


# ============================================================
# DAFTAR PMT/BC KOPEL KANDIDAT
# Nama dibuat sesuai PowerFactory. Matching juga dibuat fleksibel
# dengan normalize_name() dan compact_name().
# ============================================================

COUP_LIST = [
    "BC Sambikerep"
]

# Opsional: BC yang ingin dibaca P/Q/S-nya untuk monitoring.
# Jika tidak ingin dipakai, isi None.
BC_MONITOR_NAME = "BC Sambikerep"

BC_P_ATTRS = [
    "m:P:bus1",
    "m:Psum:bus1",
    "m:Psum",
    "m:P",
]

BC_Q_ATTRS = [
    "m:Q:bus1",
    "m:Qsum:bus1",
    "m:Qsum",
    "m:Q",
]


# ============================================================
# LINE LIST UNTUK PERHITUNGAN PIpf
# PIpf hanya dihitung dari line pada daftar ini.
# ============================================================

LINE_LIST = [
    "GRBRU-TNDES 1",
    "GRBRU-TNDES 2",
    "ALTAP-KRIAN 1",
    "ALTAP-KRIAN 2",
    "KRIAN-DARMO 1",
    "KRIAN-DARMO 2",
    "KRIAN-KLANG 1",
    "KRIAN-KLANG 2",
    "KRIAN-SWHAN1",
    "KRIAN-SWHAN2",
    "GRLMA - SMKRP 1A",
    "GRLMA - SMKRP 2A",
    "GRLMA-ALTAP 1",
    "GRLMA-ALTAP 2",
    "GRLMA-SGMDU 1",
    "GRLMA-SGMDU 2",
    "Grlma-T.Wlmar",
    "KLANG-BAMBE 1",
    "KLANG-BAMBE 2",
    "SWHAN-GNSRI 1",
    "SWHAN-GNSRI 2",
    "SWHAN-KPANG 1",
    "SWHAN-KPANG 2",
    "SWHAN-KRBAN 1",
    "SWHAN-KRBAN 2",
    "SWHAN-UDAAN 1",
    "SWHAN-UDAAN 2",
    "TNDES-SWHAN 1",
    "TNDES-SWHAN 2",
    "DARMO-WARU 1",
    "DARMO-WARU 2",
    "SGMDU-ALTAP 1",
    "SGMDU-ALTAP 2",
    "SMKRP - WARU A 1",
    "SMKRP - WARU A 2",
    "WARU-GNSRI 1",
    "WARU-GNSRI 2",
    "WARU-BDRAN 2",
    "WARU-NBDRN",
    "WARU-PTJTS",
    "WARU-RNKUT 1",
    "WARU-RNKUT 2",
    "RNKUT-HANIL",
    "RNKUT-SLILO 1",
    "RNKUT-SLILO 2",
    "SBSLN-RNKUT 1",
    "SBSLN-RNKUT 2",
    "BDRAN-NBRAN 1",
    "BDRAN-SDRJO 1B",
    "BDRAN-SDRJO 2B",
    "KLSARI-SUBSEL-I",
    "KLSARI-SUBSEL-II",
    "SLILO-KJRAN 1",
    "SLILO-KJRAN 2",
    "SLILO-KLSRI 1A",
    "SLILO-KLSRI 2A",
    "SLILO-NGAGL 1",
    "SLILO-NGAGL 2",
    "SLILO-WKRMO 1",
    "SLILO-WKRMO 2",
    "NGAGL-SIMPG 1",
    "NGAGL-SIMPG 2",
    "KJRAN-KDING 1A",
    "SDATI-NBDRAN 1",
    "SDATI-NBDRAN 2",
    "GLTMR-BKLAN 1",
    "GLTMR-KJRAN",
    "TANDES-Tx PERAK",
    "TANDES-UJUNG 1",
    "PERAK-INDSM",
    "UDAAN-GBONG 1",
    "UDAAN-GBONG 2",
    "Bklan-CH.Ujung",
    "Kding-CH.Ujung",
    "T.Perak-CH.Ujung2",
    "SURAMADU 4",
    "BANGKALAN - SAMPANG 1",
    "BANGKALAN - SAMPANG 2",
    "BKLAN-SRMDU 3B",
    "SAMPG-PMKSN 1",
    "SAMPG-PMKSN 2",
    "PMKSN-GULUK 1",
    "PMKSN-GULUK 2",
    "SMNEP-GULUK1",
    "SMNEP-GULUK2",
    "NWARU-WARU1",
    "NWARU-WARU 2",
    "NWARU-RNKUT 1",
    "NWARU-RNKUT 2",
    "NWARU-RNKUT 3",
    "NWARU-RNKUT 4",
]


# ============================================================
# DAFTAR GI / SUBSTATION YANG DIMONITOR UNTUK PIsc DAN PIv
# ============================================================

SUBSTAT_LIST = [
    "GRESIK BARU5", "KRIAN5", "GRESIK LAMA55", "KARANGPILANG5", "SAWAHAN5",
    "DARMO GRANDE5", "ALTA PRIMA5", "SEGOROMADU5", "SAMBIKEREP5", "WILMAR5",
    "BAMBE5", "GUNUNG SARI5", "PETRO KIMIA5", "WARU5", "RUNGKUT5",
    "BUDURAN5", "SURABAYA SELATAN5", "SUKOLILO5", "HANIL JAYA STEEL5",
    "JATIM TAMAN STEEL5", "KALISARI5", "WONOKROMO5", "NGAGEL5", "KENJERAN5",
    "SIDOARJO5", "NEWBUDURAN5", "SEDATI5", "SIMPANG5", "GILITIMUR5",
    "NWARU5", "TANDES5", "PERAK5", "UNDAAN5", "KUPANG5",
    "KREMBANGAN5", "UJUNG5", "INDOFOOD SUKSES MAKMUR5", "KEDINDING5",
    "BANGKALAN5", "SAMPANG5", "PAMEKASAN5", "GULUKGULUK5", "SUMENEP5",
]


# ============================================================
# BASELINE PI PER SKENARIO DAN KONDISI BEBAN
# Nilai ini digunakan sebagai pembagi PI norm.
# PIsc baseline sudah memakai:
# KRIAN5 dan NWARU5 = 50 kA, GI lain = 40 kA.
# ============================================================

BASELINE_PI = {
    "Skenario 1": {
        "Beban Rata-Rata": {"PIsc": 0.00, "PIv": 0.58, "PIpf": 4.27},
        "Beban Maksimum": {"PIsc": 0.00, "PIv": 11.40, "PIpf": 9.15},
    },
    "Skenario 2": {
        "Beban Rata-Rata": {"PIsc": 4.21, "PIv": 0.65, "PIpf": 4.60},
        "Beban Maksimum": {"PIsc": 4.21, "PIv": 2.02, "PIpf": 6.93},
    },
    "Skenario 3": {
        "Beban Rata-Rata": {"PIsc": 12.81, "PIv": 0.84, "PIpf": 2.87},
        "Beban Maksimum": {"PIsc": 12.81, "PIv": 0.43, "PIpf": 5.75},
    },
}


def get_selected_baseline(scenario_name: str, load_condition: str) -> Tuple[float, float, float]:
    if scenario_name not in BASELINE_PI:
        raise ValueError(f"Skenario tidak ditemukan: {scenario_name}")
    if load_condition not in BASELINE_PI[scenario_name]:
        raise ValueError(f"Kondisi beban tidak ditemukan: {load_condition}")

    baseline = BASELINE_PI[scenario_name][load_condition]
    return baseline["PIsc"], baseline["PIv"], baseline["PIpf"]


BASELINE_PISC, BASELINE_PIV, BASELINE_PIPF = get_selected_baseline(
    CURRENT_SCENARIO,
    LOAD_CONDITION,
)


# ============================================================
# BOBOT AHP
# Urutan kriteria: PIsc -> PIv -> PIpf
# PIsc dominan.
#
# PI Total =
# 0.637 * PIsc_norm +
# 0.258 * PIv_norm  +
# 0.105 * PIpf_norm
# ============================================================

WEIGHT_SC = 0.637
WEIGHT_V = 0.105
WEIGHT_PF = 0.258


# ============================================================
# PARAMETER PIsc
# ============================================================

DEFAULT_SCC_LIMIT_KA = 40.0
SPECIAL_SCC_LIMIT_KA = {
    "KRIAN5": 50.0,
    "NWARU5": 50.0,
}


def get_scc_limit_ka(substation_name: str) -> float:
    name = normalize_name(substation_name)
    return SPECIAL_SCC_LIMIT_KA.get(name, DEFAULT_SCC_LIMIT_KA)


# ============================================================
# PARAMETER PIv DAN VIOLATION MONITORING
# ============================================================

V_RATED_PU = 1.0
DV_LIMIT_UNDER_PU = 0.05
DV_LIMIT_OVER_PU = 0.10
VMIN = 0.95
VMAX = 1.10
MAX_LINE_LOADING_PERCENT = 80.0


# ============================================================
# PARAMETER NORMALISASI
# ============================================================

EPSILON = 1e-12
ZERO_BASELINE_PENALTY = 1_000_000.0


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_DIR = r"D:\Robbyo\S2 UGM\2026\Tesis\Hasil Uji\Final Revisi"
OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    f"{make_safe_filename(CASE_NAME)}_DS_Bus_Buka_BC_PI_Total_LINE_LIST.xlsx",
)

LOG_EACH_SCENARIO = True
SAVE_DETAIL_EACH_CONFIG = True


# ============================================================
# POWERFACTORY APP
# ============================================================

def get_app() -> Any:
    if powerfactory is None:
        raise RuntimeError("powerfactory module not available. Run this script inside DIgSILENT PowerFactory.")

    app = powerfactory.GetApplication()
    if not app:
        raise RuntimeError("Run this script inside DIgSILENT PowerFactory.")

    log(app, "===================================================")
    log(app, f"START: {CASE_NAME}")
    log(app, f"CURRENT_SCENARIO = {CURRENT_SCENARIO}")
    log(app, f"LOAD_CONDITION  = {LOAD_CONDITION}")
    log(app, f"OPEN_COUNTS     = {OPEN_COUNTS}")
    log(app, f"BAY_LIST        = {BAY_LIST}")
    log(app, f"Baseline PIsc   = {BASELINE_PISC}")
    log(app, f"Baseline PIv    = {BASELINE_PIV}")
    log(app, f"Baseline PIpf   = {BASELINE_PIPF}")
    log(app, "PIpf source: LINE_LIST only")
    log(app, "SCC Limit: KRIAN5 & NWARU5 = 50 kA, lainnya = 40 kA")
    log(app, "PI Total = 0.637*PIsc_norm + 0.258*PIpf_norm + 0.105*PIv_norm")
    log(app, "===================================================")

    return app


def safe_get_attribute(obj: Any, attr: str, default: Any = None) -> Any:
    try:
        value = obj.GetAttribute(attr)
        return default if value is None else value
    except Exception:
        return default


def safe_loc_name(obj: Any, default: str = "") -> str:
    if obj is None:
        return default
    try:
        return obj.loc_name.strip()
    except Exception:
        return default


def get_obj_key(obj: Any) -> Optional[str]:
    if obj is None:
        return None
    try:
        return obj.GetFullName()
    except Exception:
        pass
    try:
        return f"{obj.GetClassName()}::{obj.loc_name.strip()}"
    except Exception:
        return str(obj)


def get_first_attr(obj: Any, attrs: Sequence[str]) -> Any:
    for attr in attrs:
        try:
            val = obj.GetAttribute(attr)
            if val is not None:
                return val
        except Exception:
            pass
    return None


def get_parent_substation_name(obj: Any, max_depth: int = 20) -> Optional[str]:
    current = obj
    for _ in range(max_depth):
        if current is None:
            return None
        try:
            if current.GetClassName() == "ElmSubstat":
                return current.loc_name.strip()
            current = current.GetParent()
        except Exception:
            return None
    return None


def resolve_terminal(side_obj: Any) -> Any:
    if side_obj is None:
        return None
    try:
        if side_obj.GetClassName() == "ElmTerm":
            return side_obj
    except Exception:
        pass
    try:
        terminal = side_obj.cterm
        if terminal is not None:
            return terminal
    except Exception:
        pass
    return None


# ============================================================
# POWERFACTORY COMMAND
# ============================================================

def run_loadflow(app: Any) -> bool:
    app.Rebuild()
    lf = app.GetFromStudyCase("ComLdf")
    if lf is None:
        raise RuntimeError("ComLdf not found in active study case.")
    lf.iopt_net = 0
    lf.iopt_o4 = 0
    return lf.Execute() == 0


def prepare_short_circuit(app: Any) -> Any:
    sc = app.GetFromStudyCase("ComShc")
    if sc is None:
        raise RuntimeError("ComShc not found in active study case.")

    # Sesuaikan iopt_mde dengan setting studi Anda.
    # Pada script lama user memakai iopt_mde = 1.
    sc.iopt_mde = 1
    sc.Rf = 0
    sc.Xf = 0
    sc.iopt_allbus = 0
    sc.iopt_shc = "3psc"
    return sc


# ============================================================
# CACHE SWITCH / COUPLER / DS OBJECTS
# ============================================================

def build_elm_coup_lookup(app: Any) -> Dict[str, Dict[str, Any]]:
    objects = list(app.GetCalcRelevantObjects("*.ElmCoup") or [])
    normal = {}
    compact = {}

    for obj in objects:
        name = safe_loc_name(obj)
        normal.setdefault(normalize_name(name), obj)
        compact.setdefault(compact_name(name), obj)

    return {"objects": objects, "normal": normal, "compact": compact}


def find_elm_coup(lookup: Dict[str, Dict[str, Any]], target_name: str) -> Any:
    obj = lookup["normal"].get(normalize_name(target_name))
    if obj is not None:
        return obj
    return lookup["compact"].get(compact_name(target_name))


def cache_ds_pairs(app: Any, lookup: Dict[str, Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    ds_pairs = []
    missing_rows = []

    for bay in BAY_LIST:
        dsa_name = f"DSA_{bay}"
        dsb_name = f"DSB_{bay}"

        dsa_obj = find_elm_coup(lookup, dsa_name)
        dsb_obj = find_elm_coup(lookup, dsb_name)

        if dsa_obj is None or dsb_obj is None:
            missing_rows.append({
                "Bay": bay,
                "DSA Target": dsa_name,
                "DSB Target": dsb_name,
                "DSA Found": dsa_obj is not None,
                "DSB Found": dsb_obj is not None,
                "Status": "MISSING_DSA_OR_DSB",
            })
            continue

        ds_pairs.append({
            "bay": bay,
            "dsa_name": dsa_name,
            "dsb_name": dsb_name,
            "dsa_obj": dsa_obj,
            "dsb_obj": dsb_obj,
            "dsa_pf_name": safe_loc_name(dsa_obj),
            "dsb_pf_name": safe_loc_name(dsb_obj),
        })

    log(app, f"Found DS bay pairs: {len(ds_pairs)} / {len(BAY_LIST)}")
    if missing_rows:
        log(app, "Warning: beberapa pasangan DS tidak ditemukan:")
        for row in missing_rows:
            log(app, f" - {row['Bay']} | DSA found={row['DSA Found']} | DSB found={row['DSB Found']}")

    return ds_pairs, missing_rows


def cache_couplers(app: Any, lookup: Dict[str, Dict[str, Any]]) -> Tuple[Dict[str, Dict[str, Any]], List[str], List[Dict[str, Any]]]:
    couplers: Dict[str, Dict[str, Any]] = {}
    missing_rows = []

    for target_name in COUP_LIST:
        obj = find_elm_coup(lookup, target_name)
        if obj is None:
            missing_rows.append({
                "Target Coupler": target_name,
                "PowerFactory Name": None,
                "Status": "NOT_FOUND",
            })
            continue

        couplers[target_name] = {
            "obj": obj,
            "pf_name": safe_loc_name(obj),
            "orig_open": bool(obj.IsOpen()),
        }

    ordered_names = [name for name in COUP_LIST if name in couplers]

    log(app, f"Found monitored couplers: {len(ordered_names)} / {len(COUP_LIST)}")
    for name in ordered_names:
        log(app, f" - {name} | PF name: {couplers[name].get('pf_name')}")

    if missing_rows:
        log(app, "Warning: beberapa BC di COUP_LIST tidak ditemukan:")
        for row in missing_rows:
            log(app, f" - {row['Target Coupler']}")

    return couplers, ordered_names, missing_rows


def save_original_states(ds_pairs: Sequence[Dict[str, Any]], couplers: Dict[str, Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    original: Dict[str, Dict[str, Any]] = {}

    def add_obj(obj: Any) -> None:
        if obj is None:
            return
        key = get_obj_key(obj)
        if key in original:
            return
        try:
            original[key] = {"obj": obj, "is_open": bool(obj.IsOpen())}
        except Exception:
            pass

    for pair in ds_pairs:
        add_obj(pair["dsa_obj"])
        add_obj(pair["dsb_obj"])

    for info in couplers.values():
        add_obj(info["obj"])

    return original


def restore_original_states(original: Dict[str, Dict[str, Any]]) -> None:
    for info in original.values():
        obj = info["obj"]
        is_open = info["is_open"]
        try:
            if is_open:
                obj.Open()
            else:
                obj.Close()
        except Exception:
            continue


def check_binary_balance(ind: Sequence[int], min_each_side: int = MIN_BAY_PER_BUS) -> Tuple[bool, str]:
    n_bus_a = sum(ind)
    n_bus_b = len(ind) - n_bus_a

    if n_bus_a < min_each_side:
        return False, f"Rejected: Bus A only has {n_bus_a} bay(s), minimum {min_each_side}"
    if n_bus_b < min_each_side:
        return False, f"Rejected: Bus B only has {n_bus_b} bay(s), minimum {min_each_side}"

    return True, "Feasible"


def apply_ds_config(ds_pairs: Sequence[Dict[str, Any]], ind: Sequence[int]) -> None:
    for i, pair in enumerate(ds_pairs):
        if ind[i] == 1:
            pair["dsa_obj"].Close()
            pair["dsb_obj"].Open()
        else:
            pair["dsa_obj"].Open()
            pair["dsb_obj"].Close()


def apply_open_couplers(
    couplers: Dict[str, Dict[str, Any]],
    open_names: Sequence[str],
    scenario_mode: str = SCENARIO_MODE,
) -> None:
    open_set = set(open_names)

    if scenario_mode not in {"exact", "relative"}:
        raise ValueError(f"Unsupported SCENARIO_MODE: {scenario_mode}")

    for name, info in couplers.items():
        obj = info["obj"]
        if name in open_set:
            obj.Open()
        elif scenario_mode == "exact":
            obj.Close()


# ============================================================
# CACHE BUSES AND LINES
# ============================================================

def cache_buses(app: Any) -> Tuple[List[Dict[str, Any]], set, set, List[Dict[str, Any]]]:
    substations = {
        safe_loc_name(substation): substation
        for substation in app.GetCalcRelevantObjects("*.ElmSubstat")
    }
    substations_norm = {
        normalize_name(name): substation
        for name, substation in substations.items()
    }

    buses: List[Dict[str, Any]] = []
    missing_rows: List[Dict[str, Any]] = []
    monitored_substation_names = set()
    monitored_bus_keys = set()

    for sub_name in SUBSTAT_LIST:
        substation = substations.get(sub_name)
        if substation is None:
            substation = substations_norm.get(normalize_name(sub_name))

        if substation is None:
            missing_rows.append({"Substation": sub_name, "Status": "NOT_FOUND"})
            log(app, f"Warning: Substation not found: {sub_name}")
            continue

        actual_sub_name = safe_loc_name(substation) or sub_name
        monitored_substation_names.add(actual_sub_name)

        all_terms = list(substation.GetContents("*.ElmTerm", 1) or [])

        for bus in all_terms:
            usage = safe_get_attribute(bus, "iUsage")
            try:
                if int(usage) != 0:
                    continue
            except Exception:
                continue

            key = get_obj_key(bus)
            if not key or key in monitored_bus_keys:
                continue

            monitored_bus_keys.add(key)
            buses.append({
                "substation": actual_sub_name,
                "bus_name": safe_loc_name(bus),
                "obj": bus,
                "key": key,
            })

    log(app, f"Total monitored buses from SUBSTAT_LIST = {len(buses)}")
    return buses, monitored_bus_keys, monitored_substation_names, missing_rows


def get_line_terminals(line_obj: Any) -> Tuple[Any, Any]:
    cub1 = None
    cub2 = None

    try:
        cub1 = line_obj.bus1
    except Exception:
        pass
    try:
        cub2 = line_obj.bus2
    except Exception:
        pass

    if cub1 is None:
        cub1 = safe_get_attribute(line_obj, "bus1")
    if cub2 is None:
        cub2 = safe_get_attribute(line_obj, "bus2")

    return resolve_terminal(cub1), resolve_terminal(cub2)


def cache_lines(
    app: Any,
    monitored_bus_keys: set = None,
    monitored_substation_names: set = None,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    lines: List[Dict[str, Any]] = []
    missing_rows: List[Dict[str, Any]] = []
    seen_line_keys = set()

    target_line_map_normal = {normalize_name(line_name): line_name for line_name in LINE_LIST}
    target_line_map_compact = {compact_name(line_name): line_name for line_name in LINE_LIST}

    found_target_names = set()

    for line in app.GetCalcRelevantObjects("*.ElmLne"):
        line_name_pf = safe_loc_name(line)
        line_key_normal = normalize_name(line_name_pf)
        line_key_compact = compact_name(line_name_pf)

        target_name = None
        if line_key_normal in target_line_map_normal:
            target_name = target_line_map_normal[line_key_normal]
        elif line_key_compact in target_line_map_compact:
            target_name = target_line_map_compact[line_key_compact]

        if target_name is None:
            continue

        outserv = safe_get_attribute(line, "outserv", 0)
        try:
            if int(outserv) == 1:
                missing_rows.append({
                    "Target Line Name": target_name,
                    "PowerFactory Name": line_name_pf,
                    "Status": "FOUND_BUT_OUT_OF_SERVICE",
                })
                found_target_names.add(target_name)
                continue
        except Exception:
            pass

        line_obj_key = get_obj_key(line)
        if not line_obj_key or line_obj_key in seen_line_keys:
            continue

        seen_line_keys.add(line_obj_key)
        found_target_names.add(target_name)

        term1, term2 = get_line_terminals(line)
        term1_sub = get_parent_substation_name(term1) or ""
        term2_sub = get_parent_substation_name(term2) or ""

        lines.append({
            "line_name": line_name_pf,
            "line_name_target": target_name,
            "selected_side": "max_current_side",
            "obj": line,
            "bus_name": f"{safe_loc_name(term1)} / {safe_loc_name(term2)}",
            "substation": f"{term1_sub} / {term2_sub}",
        })

    for original_name in LINE_LIST:
        if original_name not in found_target_names:
            missing_rows.append({
                "Target Line Name": original_name,
                "PowerFactory Name": None,
                "Status": "NOT_FOUND",
            })

    log(app, f"Total monitored lines from LINE_LIST = {len(lines)}")
    log(app, f"Total missing/out-service lines from LINE_LIST = {len(missing_rows)}")

    if missing_rows:
        log(app, "Warning: beberapa line di LINE_LIST tidak ditemukan atau out of service:")
        for row in missing_rows:
            log(app, f" - {row['Target Line Name']} | {row['Status']}")

    return lines, missing_rows


# ============================================================
# PERFORMANCE INDEX - PIpf
# ============================================================

def calculate_pipf_total(lines: Sequence[Dict[str, Any]], save_details: bool = False) -> Tuple[float, List[Dict[str, Any]], List[str]]:
    total_pipf = 0.0
    details: List[Dict[str, Any]] = []
    violated_items: List[str] = []

    for item in lines:
        line_obj = item["obj"]

        loading = as_float(safe_get_attribute(line_obj, "m:loading", 0.0), 0.0)
        i_bus1 = as_float(safe_get_attribute(line_obj, "m:I1:bus1", 0.0), 0.0)
        i_bus2 = as_float(safe_get_attribute(line_obj, "m:I1:bus2", 0.0), 0.0)

        i_bus1 = abs(i_bus1 or 0.0)
        i_bus2 = abs(i_bus2 or 0.0)
        i_actual = max(i_bus1, i_bus2)

        i_limit = as_float(safe_get_attribute(line_obj, "c:curnom", 0.0), 0.0)
        i_limit = i_limit or 0.0

        if i_limit <= 0.0:
            pipf_i = 0.0
        else:
            pipf_i = 0.5 * ((i_actual / i_limit) ** 2)

        total_pipf += pipf_i

        line_label = item.get("line_name_target") or item.get("line_name")
        is_violated = loading is not None and loading > MAX_LINE_LOADING_PERCENT
        if is_violated:
            violated_items.append(line_label)

        if save_details:
            details.append({
                "Substation": item["substation"],
                "Bus": item["bus_name"],
                "Line": item["line_name"],
                "Target Line Name": item.get("line_name_target"),
                "Selected Side": "Max(bus1,bus2)",
                "Loading (%)": safe_round(loading, 6),
                "Loading Limit (%)": MAX_LINE_LOADING_PERCENT,
                "Loading Violated": is_violated,
                "I bus1 (kA)": safe_round(i_bus1, 6),
                "I bus2 (kA)": safe_round(i_bus2, 6),
                "I actual max (kA)": safe_round(i_actual, 6),
                "I limit (kA)": safe_round(i_limit, 6),
                "PIpf_i = 0.5*(Iactual/Ilimit)^2": safe_round(pipf_i, 8),
            })

    return total_pipf, details, sorted(set(violated_items))


# ============================================================
# PERFORMANCE INDEX - PIsc
# ============================================================

def calculate_pisc_total(sc: Any, buses: Sequence[Dict[str, Any]], save_details: bool = False) -> Tuple[Optional[float], List[Dict[str, Any]], List[str], str]:
    total_pisc = 0.0
    details: List[Dict[str, Any]] = []
    violated_subs: List[str] = []

    for item in buses:
        substation_name = item["substation"]
        bus = item["obj"]
        sc.shcobj = bus

        if sc.Execute() != 0:
            return None, details, violated_subs, f"SC_EXECUTE_FAILED: {item['substation']} / {item['bus_name']}"

        isc = as_float(safe_get_attribute(bus, "m:Ikss", None), None)
        if isc is None:
            isc = as_float(safe_get_attribute(bus, "m:ikss", None), None)

        if isc is None:
            return None, details, violated_subs, f"SC_RESULT_MISSING: {item['substation']} / {item['bus_name']}"

        scc_limit_ka = get_scc_limit_ka(substation_name)

        if isc > scc_limit_ka:
            pisc_i = (isc / scc_limit_ka) ** 2
            violation = True
            violated_subs.append(substation_name)
        else:
            pisc_i = 0.0
            violation = False

        total_pisc += pisc_i

        if save_details:
            details.append({
                "Substation": item["substation"],
                "Bus": item["bus_name"],
                "Isc / Ikss (kA)": safe_round(isc, 6),
                "SCC Limit Used (kA)": safe_round(scc_limit_ka, 6),
                "PIsc_i = (Isc/SCC Limit)^2 if Isc>Limit": safe_round(pisc_i, 8),
                "Violation": violation,
            })

    return total_pisc, details, sorted(set(violated_subs)), "OK"


# ============================================================
# PERFORMANCE INDEX - PIv
# ============================================================

def calculate_piv_total(buses: Sequence[Dict[str, Any]], save_details: bool = False) -> Tuple[Optional[float], List[Dict[str, Any]], List[str], str]:
    total_piv = 0.0
    details: List[Dict[str, Any]] = []
    violated_subs: List[str] = []

    for item in buses:
        sub_name = item["substation"]
        bus_name = item["bus_name"]
        bus_obj = item["obj"]

        vpu = as_float(safe_get_attribute(bus_obj, "m:u", None), None)
        vkv = as_float(safe_get_attribute(bus_obj, "m:Ul", None), None)

        if vpu is None:
            return None, details, violated_subs, f"VOLTAGE_RESULT_MISSING: {sub_name} / {bus_name}"

        deviation = abs(vpu - V_RATED_PU)

        if vpu < V_RATED_PU:
            dv_limit = DV_LIMIT_UNDER_PU
        elif vpu > V_RATED_PU:
            dv_limit = DV_LIMIT_OVER_PU
        else:
            dv_limit = DV_LIMIT_UNDER_PU

        piv_i = 0.5 * ((deviation / dv_limit) ** 2)
        total_piv += piv_i

        if vpu < VMIN:
            status = "LOW"
            violated_subs.append(sub_name)
        elif vpu > VMAX:
            status = "HIGH"
            violated_subs.append(sub_name)
        else:
            status = "NORMAL"

        if save_details:
            details.append({
                "Substation": sub_name,
                "Bus": bus_name,
                "Vpu": safe_round(vpu, 6),
                "VkV": safe_round(vkv, 3) if vkv is not None else None,
                "Vrated_pu": V_RATED_PU,
                "DeltaVlim_pu": dv_limit,
                "Deviation": safe_round(deviation, 8),
                "PIv_i = 0.5*(Deviation/DeltaVlim)^2": safe_round(piv_i, 8),
                "Status": status,
            })

    return total_piv, details, sorted(set(violated_subs)), "OK"


# ============================================================
# NORMALISASI DAN PI TOTAL
# ============================================================

def normalize_by_baseline(value: Optional[float], baseline: float) -> Optional[float]:
    if value is None:
        return None
    if abs(baseline) <= EPSILON:
        if abs(value) <= EPSILON:
            return 1.0
        return ZERO_BASELINE_PENALTY
    return value / baseline


def calculate_pi_total(pisc_total: float, piv_total: float, pipf_total: float) -> Tuple[
    Optional[float], Optional[float], Optional[float], Optional[float], Optional[float], Optional[float], Optional[float]
]:
    pisc_norm = normalize_by_baseline(pisc_total, BASELINE_PISC)
    piv_norm = normalize_by_baseline(piv_total, BASELINE_PIV)
    pipf_norm = normalize_by_baseline(pipf_total, BASELINE_PIPF)

    if pisc_norm is None or piv_norm is None or pipf_norm is None:
        return None, pisc_norm, piv_norm, pipf_norm, None, None, None

    pisc_contribution = WEIGHT_SC * pisc_norm
    piv_contribution = WEIGHT_V * piv_norm
    pipf_contribution = WEIGHT_PF * pipf_norm
    pi_total = pisc_contribution + piv_contribution + pipf_contribution

    return pi_total, pisc_norm, piv_norm, pipf_norm, pisc_contribution, piv_contribution, pipf_contribution


# ============================================================
# FORMAT DS CONFIG
# ============================================================

def format_binary(ind: Sequence[int]) -> str:
    return "".join(str(bit) for bit in ind)


def build_config_detail(ind: Sequence[int]) -> List[Dict[str, Any]]:
    rows = []
    for i, bay in enumerate(BAY_LIST):
        bit = ind[i]
        if bit == 1:
            bus = "Bus A"
            dsa_state = "CLOSE"
            dsb_state = "OPEN"
        else:
            bus = "Bus B"
            dsa_state = "OPEN"
            dsb_state = "CLOSE"

        rows.append({
            "No": i + 1,
            "Bay": bay,
            "Bit": bit,
            "Bus": bus,
            "DSA": dsa_state,
            "DSB": dsb_state,
        })
    return rows


def read_bc_monitor(couplers: Dict[str, Dict[str, Any]]) -> Tuple[Optional[float], Optional[float], Optional[float]]:
    if not BC_MONITOR_NAME or BC_MONITOR_NAME not in couplers:
        return None, None, None

    bc_obj = couplers[BC_MONITOR_NAME]["obj"]
    p_bc = as_float(get_first_attr(bc_obj, BC_P_ATTRS), None)
    q_bc = as_float(get_first_attr(bc_obj, BC_Q_ATTRS), None)

    if p_bc is not None and q_bc is not None:
        s_bc = math.sqrt(p_bc * p_bc + q_bc * q_bc)
    else:
        s_bc = None

    return p_bc, q_bc, s_bc


# ============================================================
# EVALUASI SETIAP KOMBINASI DS + BUKA PMT/BC KOPEL
# ============================================================

def evaluate_combination(
    app: Any,
    ds_pairs: Sequence[Dict[str, Any]],
    couplers: Dict[str, Dict[str, Any]],
    original_state: Dict[str, Dict[str, Any]],
    lines: Sequence[Dict[str, Any]],
    buses: Sequence[Dict[str, Any]],
    ds_ind: Sequence[int],
    open_names: Sequence[str],
    save_details: bool = False,
) -> Dict[str, Any]:
    valid_binary, binary_reason = check_binary_balance(ds_ind)
    if not valid_binary:
        return {
            "status": "BINARY_CONSTRAINT",
            "reject_reason": binary_reason,
            "pi_total": REJECT_PI_TOTAL,
            "pisc_total": None,
            "piv_total": None,
            "pipf_total": None,
            "pisc_norm": None,
            "piv_norm": None,
            "pipf_norm": None,
            "pisc_contribution": None,
            "piv_contribution": None,
            "pipf_contribution": None,
            "p_bc": None,
            "q_bc": None,
            "s_bc": None,
            "violated": True,
            "viol_sc": [],
            "viol_v": [],
            "viol_pf": [],
            "line_details": [],
            "v_details": [],
            "sc_details": [],
        }

    restore_original_states(original_state)

    try:
        apply_ds_config(ds_pairs, ds_ind)
        apply_open_couplers(couplers, open_names, SCENARIO_MODE)

        if not run_loadflow(app):
            return {
                "status": "LOADFLOW_FAILED",
                "reject_reason": "Load flow failed",
                "pi_total": REJECT_PI_TOTAL,
                "pisc_total": None,
                "piv_total": None,
                "pipf_total": None,
                "pisc_norm": None,
                "piv_norm": None,
                "pipf_norm": None,
                "pisc_contribution": None,
                "piv_contribution": None,
                "pipf_contribution": None,
                "p_bc": None,
                "q_bc": None,
                "s_bc": None,
                "violated": True,
                "viol_sc": [],
                "viol_v": [],
                "viol_pf": [],
                "line_details": [],
                "v_details": [],
                "sc_details": [],
            }

        p_bc, q_bc, s_bc = read_bc_monitor(couplers)

        pipf_total, line_details, viol_pf = calculate_pipf_total(lines, save_details=save_details)
        piv_total, v_details, viol_v, piv_status = calculate_piv_total(buses, save_details=save_details)

        if piv_total is None:
            return {
                "status": piv_status,
                "reject_reason": piv_status,
                "pi_total": REJECT_PI_TOTAL,
                "pisc_total": None,
                "piv_total": None,
                "pipf_total": pipf_total,
                "pisc_norm": None,
                "piv_norm": None,
                "pipf_norm": None,
                "pisc_contribution": None,
                "piv_contribution": None,
                "pipf_contribution": None,
                "p_bc": p_bc,
                "q_bc": q_bc,
                "s_bc": s_bc,
                "violated": True,
                "viol_sc": [],
                "viol_v": viol_v,
                "viol_pf": viol_pf,
                "line_details": line_details,
                "v_details": v_details,
                "sc_details": [],
            }

        sc = prepare_short_circuit(app)
        pisc_total, sc_details, viol_sc, pisc_status = calculate_pisc_total(sc, buses, save_details=save_details)

        if pisc_total is None:
            return {
                "status": pisc_status,
                "reject_reason": pisc_status,
                "pi_total": REJECT_PI_TOTAL,
                "pisc_total": None,
                "piv_total": piv_total,
                "pipf_total": pipf_total,
                "pisc_norm": None,
                "piv_norm": None,
                "pipf_norm": None,
                "pisc_contribution": None,
                "piv_contribution": None,
                "pipf_contribution": None,
                "p_bc": p_bc,
                "q_bc": q_bc,
                "s_bc": s_bc,
                "violated": True,
                "viol_sc": viol_sc,
                "viol_v": viol_v,
                "viol_pf": viol_pf,
                "line_details": line_details,
                "v_details": v_details,
                "sc_details": sc_details,
            }

        (
            pi_total,
            pisc_norm,
            piv_norm,
            pipf_norm,
            pisc_contribution,
            piv_contribution,
            pipf_contribution,
        ) = calculate_pi_total(
            pisc_total=pisc_total,
            piv_total=piv_total,
            pipf_total=pipf_total,
        )

        violated = len(set(viol_sc + viol_v + viol_pf)) > 0

        return {
            "status": "OK",
            "reject_reason": "Feasible",
            "pi_total": pi_total,
            "pisc_total": pisc_total,
            "piv_total": piv_total,
            "pipf_total": pipf_total,
            "pisc_norm": pisc_norm,
            "piv_norm": piv_norm,
            "pipf_norm": pipf_norm,
            "pisc_contribution": pisc_contribution,
            "piv_contribution": piv_contribution,
            "pipf_contribution": pipf_contribution,
            "p_bc": p_bc,
            "q_bc": q_bc,
            "s_bc": s_bc,
            "violated": violated,
            "viol_sc": sorted(set(viol_sc)),
            "viol_v": sorted(set(viol_v)),
            "viol_pf": sorted(set(viol_pf)),
            "line_details": line_details,
            "v_details": v_details,
            "sc_details": sc_details,
        }

    except Exception as exc:
        return {
            "status": f"ERROR: {exc}",
            "reject_reason": str(exc),
            "pi_total": REJECT_PI_TOTAL,
            "pisc_total": None,
            "piv_total": None,
            "pipf_total": None,
            "pisc_norm": None,
            "piv_norm": None,
            "pipf_norm": None,
            "pisc_contribution": None,
            "piv_contribution": None,
            "pipf_contribution": None,
            "p_bc": None,
            "q_bc": None,
            "s_bc": None,
            "violated": True,
            "viol_sc": [],
            "viol_v": [],
            "viol_pf": [],
            "line_details": [],
            "v_details": [],
            "sc_details": [],
        }

    finally:
        restore_original_states(original_state)


def build_ranking(df_all: pd.DataFrame, score_column: str) -> pd.DataFrame:
    if df_all.empty or score_column not in df_all.columns:
        return pd.DataFrame()

    df_valid = df_all[df_all[score_column].notna()]
    if df_valid.empty:
        return pd.DataFrame()

    return df_valid.sort_values(
        by=[score_column, "PIsc Total", "PIv Total", "PIpf Total"],
        ascending=[True, True, True, True],
    ).reset_index(drop=True)


# ============================================================
# MAIN PROGRAM
# ============================================================

def run_optimization_ds_bus_and_bc() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    app = get_app()
    t0 = time.perf_counter()

    coup_lookup = build_elm_coup_lookup(app)
    ds_pairs, missing_ds_rows = cache_ds_pairs(app, coup_lookup)
    couplers, coupler_names, missing_coupler_rows = cache_couplers(app, coup_lookup)
    buses, monitored_bus_keys, monitored_substation_names, missing_sub_rows = cache_buses(app)
    lines, missing_line_rows = cache_lines(app, monitored_bus_keys, monitored_substation_names)

    if len(ds_pairs) != len(BAY_LIST):
        raise RuntimeError("Tidak semua pasangan DSA/DSB ditemukan. Cek sheet/log missing DS.")
    if not coupler_names:
        raise RuntimeError("No couplers found from COUP_LIST.")
    if not buses:
        raise RuntimeError("No buses found from SUBSTAT_LIST.")
    if not lines:
        raise RuntimeError("No lines found from LINE_LIST.")

    original_state = save_original_states(ds_pairs, couplers)

    all_rows: List[Dict[str, Any]] = []
    detail_line_rows: List[Dict[str, Any]] = []
    detail_sc_rows: List[Dict[str, Any]] = []
    detail_v_rows: List[Dict[str, Any]] = []

    best_score = float("inf")
    best_ds_ind: Optional[List[int]] = None
    best_open_names: Optional[List[str]] = None
    best_result: Optional[Dict[str, Any]] = None

    ds_configs = list(product([0, 1], repeat=len(BAY_LIST)))
    bc_combos = []
    for n_open in OPEN_COUNTS:
        if n_open <= len(coupler_names):
            bc_combos.extend(list(combinations(coupler_names, n_open)))

    total_eval = len(ds_configs) * len(bc_combos)

    log(app, "===================================================")
    log(app, f"Total DS config       : {len(ds_configs)}")
    log(app, f"Total BC combination  : {len(bc_combos)}")
    log(app, f"Total evaluation      : {total_eval}")
    log(app, "===================================================")

    eval_no = 1

    try:
        for ds_idx, ds_tuple in enumerate(ds_configs, start=1):
            ds_ind = list(ds_tuple)
            ds_binary = format_binary(ds_ind)

            for bc_combo in bc_combos:
                open_names = list(bc_combo)
                open_label = ", ".join(open_names)

                if LOG_EACH_SCENARIO:
                    log(app, f"[{eval_no}/{total_eval}] DS={ds_binary} | OPEN BC={open_label}")

                result = evaluate_combination(
                    app=app,
                    ds_pairs=ds_pairs,
                    couplers=couplers,
                    original_state=original_state,
                    lines=lines,
                    buses=buses,
                    ds_ind=ds_ind,
                    open_names=open_names,
                    save_details=SAVE_DETAIL_EACH_CONFIG,
                )

                pi_total = result.get("pi_total")
                pisc_total = result.get("pisc_total")
                piv_total = result.get("piv_total")
                pipf_total = result.get("pipf_total")

                pisc_norm = result.get("pisc_norm")
                piv_norm = result.get("piv_norm")
                pipf_norm = result.get("pipf_norm")

                pisc_contribution = result.get("pisc_contribution")
                piv_contribution = result.get("piv_contribution")
                pipf_contribution = result.get("pipf_contribution")

                all_rows.append({
                    "Eval No": eval_no,
                    "Current Scenario": CURRENT_SCENARIO,
                    "Load Condition": LOAD_CONDITION,
                    "DS Config No": ds_idx,
                    "DS Config List": str(ds_ind),
                    "DS Config Binary": ds_binary,
                    "N Open BC": len(open_names),
                    "Opened BC": open_label,

                    "PI Total": safe_round(pi_total, 8),

                    "PIsc Total": safe_round(pisc_total, 8),
                    "PIsc Baseline": BASELINE_PISC,
                    f"PIsc Norm = PIsc Total / {BASELINE_PISC}": safe_round(pisc_norm, 8),
                    "Weight PIsc": WEIGHT_SC,
                    f"PIsc Contribution = {WEIGHT_SC}*PIsc Norm": safe_round(pisc_contribution, 8),

                    "PIv Total": safe_round(piv_total, 8),
                    "PIv Baseline": BASELINE_PIV,
                    f"PIv Norm = PIv Total / {BASELINE_PIV}": safe_round(piv_norm, 8),
                    "Weight PIv": WEIGHT_V,
                    f"PIv Contribution = {WEIGHT_V}*PIv Norm": safe_round(piv_contribution, 8),

                    "PIpf Total": safe_round(pipf_total, 8),
                    "PIpf Baseline": BASELINE_PIPF,
                    f"PIpf Norm = PIpf Total / {BASELINE_PIPF}": safe_round(pipf_norm, 8),
                    "Weight PIpf": WEIGHT_PF,
                    f"PIpf Contribution = {WEIGHT_PF}*PIpf Norm": safe_round(pipf_contribution, 8),

                    "P Monitor BC (MW)": safe_round(result.get("p_bc"), 6),
                    "Q Monitor BC (Mvar)": safe_round(result.get("q_bc"), 6),
                    "S Monitor BC (MVA)": safe_round(result.get("s_bc"), 6),
                    "BC Monitor Name": BC_MONITOR_NAME,

                    "Violated": result.get("violated"),
                    "Viol SC": ", ".join(result.get("viol_sc", [])),
                    "Viol V": ", ".join(result.get("viol_v", [])),
                    "Viol PF": ", ".join(result.get("viol_pf", [])),
                    "Status": result.get("status"),
                    "Reject Reason": result.get("reject_reason"),
                })

                if SAVE_DETAIL_EACH_CONFIG:
                    for row in result.get("sc_details", []):
                        detail_sc_rows.append({
                            "Eval No": eval_no,
                            "DS Binary": ds_binary,
                            "Opened BC": open_label,
                            "PIsc Total": safe_round(pisc_total, 8),
                            "PIsc Norm": safe_round(pisc_norm, 8),
                            "PIsc Contribution": safe_round(pisc_contribution, 8),
                            **row,
                        })

                    for row in result.get("v_details", []):
                        detail_v_rows.append({
                            "Eval No": eval_no,
                            "DS Binary": ds_binary,
                            "Opened BC": open_label,
                            "PIv Total": safe_round(piv_total, 8),
                            "PIv Norm": safe_round(piv_norm, 8),
                            "PIv Contribution": safe_round(piv_contribution, 8),
                            **row,
                        })

                    for row in result.get("line_details", []):
                        detail_line_rows.append({
                            "Eval No": eval_no,
                            "DS Binary": ds_binary,
                            "Opened BC": open_label,
                            "PIpf Total": safe_round(pipf_total, 8),
                            "PIpf Norm": safe_round(pipf_norm, 8),
                            "PIpf Contribution": safe_round(pipf_contribution, 8),
                            **row,
                        })

                if result.get("status") == "OK" and pi_total is not None and pi_total < best_score:
                    best_score = pi_total
                    best_ds_ind = ds_ind[:]
                    best_open_names = open_names[:]
                    best_result = result.copy()
                    log(app, f"NEW BEST PI TOTAL = {best_score:.8f} | DS={ds_binary} | OPEN={open_label}")

                if eval_no % 25 == 0 or eval_no == total_eval:
                    best_bin = format_binary(best_ds_ind) if best_ds_ind is not None else None
                    best_open = ", ".join(best_open_names) if best_open_names else None
                    log(app, f"Progress {eval_no}/{total_eval} | Best={best_score:.8f} | DS={best_bin} | OPEN={best_open}")

                eval_no += 1

    finally:
        restore_original_states(original_state)

    elapsed = time.perf_counter() - t0

    df_all = pd.DataFrame(all_rows)
    df_rank = build_ranking(df_all, "PI Total")
    if not df_rank.empty:
        df_rank.insert(0, "Rank", range(1, len(df_rank) + 1))

    best_label = ", ".join(best_open_names) if best_open_names else None
    best_binary = format_binary(best_ds_ind) if best_ds_ind else None

    best_summary = {
        "Case": CASE_NAME,
        "Current Scenario": CURRENT_SCENARIO,
        "Load Condition": LOAD_CONDITION,
        "Scenario Mode": SCENARIO_MODE,
        "Open Counts": str(OPEN_COUNTS),
        "Best DS Config List": str(best_ds_ind) if best_ds_ind is not None else None,
        "Best DS Config Binary": best_binary,
        "Best Opened BC": best_label,
        "Best N Open BC": len(best_open_names) if best_open_names else None,
        "Best PI Total": safe_round(best_score, 8) if best_result else None,
        "Best PIsc Total": safe_round(best_result.get("pisc_total"), 8) if best_result else None,
        "Best PIv Total": safe_round(best_result.get("piv_total"), 8) if best_result else None,
        "Best PIpf Total": safe_round(best_result.get("pipf_total"), 8) if best_result else None,
        "Baseline PIsc": BASELINE_PISC,
        "Baseline PIv": BASELINE_PIV,
        "Baseline PIpf": BASELINE_PIPF,
        "Weight PIsc": WEIGHT_SC,
        "Weight PIv": WEIGHT_V,
        "Weight PIpf": WEIGHT_PF,
        "Default SCC Limit (kA)": DEFAULT_SCC_LIMIT_KA,
        "Special SCC Limit": "KRIAN5=50 kA; NWARU5=50 kA",
        "Total DS Config": len(ds_configs),
        "Total BC Combination": len(bc_combos),
        "Total Evaluation": total_eval,
        "Total PIpf Lines Found": len(lines),
        "Total Missing/Out-service Lines": len(missing_line_rows),
        "Execution Time (s)": safe_round(elapsed, 2),
    }

    df_best = pd.DataFrame([best_summary])
    df_best_cfg = pd.DataFrame(build_config_detail(best_ds_ind)) if best_ds_ind else pd.DataFrame()
    df_detail_sc = pd.DataFrame(detail_sc_rows)
    df_detail_v = pd.DataFrame(detail_v_rows)
    df_detail_line = pd.DataFrame(detail_line_rows)

    df_missing_ds = pd.DataFrame(missing_ds_rows)
    df_missing_coupler = pd.DataFrame(missing_coupler_rows)
    df_missing_sub = pd.DataFrame(missing_sub_rows)
    df_missing_line = pd.DataFrame(missing_line_rows)

    reject_summary = pd.DataFrame()
    if not df_all.empty:
        reject_summary = (
            df_all.groupby(["Status", "Reject Reason", "Violated"], dropna=False)
            .size()
            .reset_index(name="Count")
            .sort_values(["Status", "Reject Reason", "Violated"])
        )

    df_param = pd.DataFrame([
        {"Parameter": "CURRENT_SCENARIO", "Value": CURRENT_SCENARIO},
        {"Parameter": "LOAD_CONDITION", "Value": LOAD_CONDITION},
        {"Parameter": "OPEN_COUNTS", "Value": str(OPEN_COUNTS)},
        {"Parameter": "SCENARIO_MODE", "Value": SCENARIO_MODE},
        {"Parameter": "BAY_LIST", "Value": ", ".join(BAY_LIST)},
        {"Parameter": "COUP_LIST", "Value": ", ".join(COUP_LIST)},
        {"Parameter": "BASELINE_PISC", "Value": BASELINE_PISC},
        {"Parameter": "BASELINE_PIV", "Value": BASELINE_PIV},
        {"Parameter": "BASELINE_PIPF", "Value": BASELINE_PIPF},
        {"Parameter": "WEIGHT_SC", "Value": WEIGHT_SC},
        {"Parameter": "WEIGHT_V", "Value": WEIGHT_V},
        {"Parameter": "WEIGHT_PF", "Value": WEIGHT_PF},
        {"Parameter": "DEFAULT_SCC_LIMIT_KA", "Value": DEFAULT_SCC_LIMIT_KA},
        {"Parameter": "SPECIAL_SCC_LIMIT_KA", "Value": str(SPECIAL_SCC_LIMIT_KA)},
        {"Parameter": "V_RATED_PU", "Value": V_RATED_PU},
        {"Parameter": "DV_LIMIT_UNDER_PU", "Value": DV_LIMIT_UNDER_PU},
        {"Parameter": "DV_LIMIT_OVER_PU", "Value": DV_LIMIT_OVER_PU},
        {"Parameter": "MAX_LINE_LOADING_PERCENT", "Value": MAX_LINE_LOADING_PERCENT},
        {"Parameter": "MIN_BAY_PER_BUS", "Value": MIN_BAY_PER_BUS},
        {"Parameter": "REJECT_PI_TOTAL", "Value": REJECT_PI_TOTAL},
        {"Parameter": "OUTPUT_FILE", "Value": OUTPUT_FILE},
    ])

    with pd.ExcelWriter(OUTPUT_FILE) as writer:
        df_all.to_excel(writer, sheet_name="All_Configurations", index=False)
        df_rank.to_excel(writer, sheet_name="Ranking_PI_Total", index=False)
        df_best.to_excel(writer, sheet_name="Best_Result", index=False)
        if not df_best_cfg.empty:
            df_best_cfg.to_excel(writer, sheet_name="Best_DS_Config", index=False)
        if not df_detail_sc.empty:
            df_detail_sc.to_excel(writer, sheet_name="Detail_PIsc", index=False)
        if not df_detail_v.empty:
            df_detail_v.to_excel(writer, sheet_name="Detail_PIv", index=False)
        if not df_detail_line.empty:
            df_detail_line.to_excel(writer, sheet_name="Detail_PIpf_LINE_LIST", index=False)
        if not reject_summary.empty:
            reject_summary.to_excel(writer, sheet_name="Reject_Summary", index=False)
        if not df_missing_ds.empty:
            df_missing_ds.to_excel(writer, sheet_name="Missing_DS", index=False)
        if not df_missing_coupler.empty:
            df_missing_coupler.to_excel(writer, sheet_name="Missing_Coupler", index=False)
        if not df_missing_sub.empty:
            df_missing_sub.to_excel(writer, sheet_name="Missing_Substation", index=False)
        if not df_missing_line.empty:
            df_missing_line.to_excel(writer, sheet_name="Missing_LINE_LIST", index=False)
        df_param.to_excel(writer, sheet_name="Parameters", index=False)

    log(app, "===================================================")
    log(app, "FINISHED")
    log(app, f"Saved Excel : {OUTPUT_FILE}")
    log(app, f"Execution   : {elapsed:.2f} s")
    if best_result:
        log(app, f"Best DS Binary : {best_binary}")
        log(app, f"Best Opened BC : {best_label}")
        log(app, f"Best PI Total  : {best_score:.8f}")
    else:
        log(app, "Best result not found.")
    log(app, "Kondisi DS dan BC sudah dikembalikan ke kondisi awal.")
    log(app, "===================================================")


if __name__ == "__main__":
    run_optimization_ds_bus_and_bc()


Ambil nilai PI total saat Kopel sambikerep dan sawahan sudah dibuka

In [15]:
import math
import os
import time
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    import powerfactory
except ImportError:
    powerfactory = None


# ============================================================
# SCRIPT MANUAL CONFIGURATION READER
# ============================================================
# Tujuan:
# - Tidak mengubah posisi DS Bus.
# - Tidak membuka / menutup BC atau kopel.
# - Membaca kondisi jaringan yang sedang aktif di DIgSILENT PowerFactory.
# - Menjalankan Load Flow untuk PIv dan PIpf.
# - Menjalankan Short Circuit untuk PIsc.
# - Menghitung PI Total dan menyimpan hasil ke Excel.
#
# Cara pakai:
# 1. Atur konfigurasi DS Bus / BC secara manual di PowerFactory.
# 2. Pastikan study case yang benar sudah aktif.
# 3. Jalankan script ini dari PowerFactory.
# ============================================================


# ============================================================
# PILIH SKENARIO DAN KONDISI BEBAN
# Pastikan study case aktif di PowerFactory sesuai pilihan ini.
# ============================================================

CURRENT_SCENARIO = 'Skenario 3'
LOAD_CONDITION = 'Beban Maksimum'

# Pilihan:
# CURRENT_SCENARIO = "Skenario 1", "Skenario 2", "Skenario 3"
# LOAD_CONDITION  = "Beban Rata-Rata", "Beban Maksimum"

CASE_NAME = f"{CURRENT_SCENARIO} - {LOAD_CONDITION}"


# ============================================================
# BC MONITORING SAJA
# Tidak ada proses buka/tutup BC.
# Nilai P/Q/S dibaca dari kondisi BC saat ini.
# Jika tidak ingin monitor BC, isi None.
# ============================================================

BC_MONITOR_NAMES = [
    "BC Sambikerep",
    "BC Sawahan",
]

BC_P_ATTRS = ['m:P:bus1', 'm:Psum:bus1', 'm:Psum', 'm:P']

BC_Q_ATTRS = ['m:Q:bus1', 'm:Qsum:bus1', 'm:Qsum', 'm:Q']


# ============================================================
# LINE LIST UNTUK PERHITUNGAN PIpf
# PIpf hanya dihitung dari line pada daftar ini.
# ============================================================

LINE_LIST = ['GRBRU-TNDES 1',
 'GRBRU-TNDES 2',
 'ALTAP-KRIAN 1',
 'ALTAP-KRIAN 2',
 'KRIAN-DARMO 1',
 'KRIAN-DARMO 2',
 'KRIAN-KLANG 1',
 'KRIAN-KLANG 2',
 'KRIAN-SWHAN1',
 'KRIAN-SWHAN2',
 'GRLMA - SMKRP 1A',
 'GRLMA - SMKRP 2A',
 'GRLMA-ALTAP 1',
 'GRLMA-ALTAP 2',
 'GRLMA-SGMDU 1',
 'GRLMA-SGMDU 2',
 'Grlma-T.Wlmar',
 'KLANG-BAMBE 1',
 'KLANG-BAMBE 2',
 'SWHAN-GNSRI 1',
 'SWHAN-GNSRI 2',
 'SWHAN-KPANG 1',
 'SWHAN-KPANG 2',
 'SWHAN-KRBAN 1',
 'SWHAN-KRBAN 2',
 'SWHAN-UDAAN 1',
 'SWHAN-UDAAN 2',
 'TNDES-SWHAN 1',
 'TNDES-SWHAN 2',
 'DARMO-WARU 1',
 'DARMO-WARU 2',
 'SGMDU-ALTAP 1',
 'SGMDU-ALTAP 2',
 'SMKRP - WARU A 1',
 'SMKRP - WARU A 2',
 'WARU-GNSRI 1',
 'WARU-GNSRI 2',
 'WARU-BDRAN 2',
 'WARU-NBDRN',
 'WARU-PTJTS',
 'WARU-RNKUT 1',
 'WARU-RNKUT 2',
 'RNKUT-HANIL',
 'RNKUT-SLILO 1',
 'RNKUT-SLILO 2',
 'SBSLN-RNKUT 1',
 'SBSLN-RNKUT 2',
 'BDRAN-NBRAN 1',
 'BDRAN-SDRJO 1B',
 'BDRAN-SDRJO 2B',
 'KLSARI-SUBSEL-I',
 'KLSARI-SUBSEL-II',
 'SLILO-KJRAN 1',
 'SLILO-KJRAN 2',
 'SLILO-KLSRI 1A',
 'SLILO-KLSRI 2A',
 'SLILO-NGAGL 1',
 'SLILO-NGAGL 2',
 'SLILO-WKRMO 1',
 'SLILO-WKRMO 2',
 'NGAGL-SIMPG 1',
 'NGAGL-SIMPG 2',
 'KJRAN-KDING 1A',
 'SDATI-NBDRAN 1',
 'SDATI-NBDRAN 2',
 'GLTMR-BKLAN 1',
 'GLTMR-KJRAN',
 'TANDES-Tx PERAK',
 'TANDES-UJUNG 1',
 'PERAK-INDSM',
 'UDAAN-GBONG 1',
 'UDAAN-GBONG 2',
 'Bklan-CH.Ujung',
 'Kding-CH.Ujung',
 'T.Perak-CH.Ujung2',
 'SURAMADU 4',
 'BANGKALAN - SAMPANG 1',
 'BANGKALAN - SAMPANG 2',
 'BKLAN-SRMDU 3B',
 'SAMPG-PMKSN 1',
 'SAMPG-PMKSN 2',
 'PMKSN-GULUK 1',
 'PMKSN-GULUK 2',
 'SMNEP-GULUK1',
 'SMNEP-GULUK2',
 'NWARU-WARU1',
 'NWARU-WARU 2',
 'NWARU-RNKUT 1',
 'NWARU-RNKUT 2',
 'NWARU-RNKUT 3',
 'NWARU-RNKUT 4']


# ============================================================
# DAFTAR GI / SUBSTATION YANG DIMONITOR UNTUK PIsc DAN PIv
# ============================================================

SUBSTAT_LIST = ['GRESIK BARU5',
 'KRIAN5',
 'GRESIK LAMA55',
 'KARANGPILANG5',
 'SAWAHAN5',
 'DARMO GRANDE5',
 'ALTA PRIMA5',
 'SEGOROMADU5',
 'SAMBIKEREP5',
 'WILMAR5',
 'BAMBE5',
 'GUNUNG SARI5',
 'PETRO KIMIA5',
 'WARU5',
 'RUNGKUT5',
 'BUDURAN5',
 'SURABAYA SELATAN5',
 'SUKOLILO5',
 'HANIL JAYA STEEL5',
 'JATIM TAMAN STEEL5',
 'KALISARI5',
 'WONOKROMO5',
 'NGAGEL5',
 'KENJERAN5',
 'SIDOARJO5',
 'NEWBUDURAN5',
 'SEDATI5',
 'SIMPANG5',
 'GILITIMUR5',
 'NWARU5',
 'TANDES5',
 'PERAK5',
 'UNDAAN5',
 'KUPANG5',
 'KREMBANGAN5',
 'UJUNG5',
 'INDOFOOD SUKSES MAKMUR5',
 'KEDINDING5',
 'BANGKALAN5',
 'SAMPANG5',
 'PAMEKASAN5',
 'GULUKGULUK5',
 'SUMENEP5']


# ============================================================
# BASELINE PI PER SKENARIO DAN KONDISI BEBAN
# Nilai ini digunakan sebagai pembagi PI norm.
# ============================================================

BASELINE_PI = {'Skenario 1': {'Beban Maksimum': {'PIpf': 9.15, 'PIsc': 0.0, 'PIv': 11.4},
                'Beban Rata-Rata': {'PIpf': 4.27, 'PIsc': 0.0, 'PIv': 0.58}},
 'Skenario 2': {'Beban Maksimum': {'PIpf': 6.93, 'PIsc': 4.21, 'PIv': 2.02},
                'Beban Rata-Rata': {'PIpf': 4.6, 'PIsc': 4.21, 'PIv': 0.65}},
 'Skenario 3': {'Beban Maksimum': {'PIpf': 5.75, 'PIsc': 12.81, 'PIv': 0.43},
                'Beban Rata-Rata': {'PIpf': 2.87, 'PIsc': 12.81, 'PIv': 0.84}}}


def get_selected_baseline(scenario_name: str, load_condition: str) -> Tuple[float, float, float]:
    if scenario_name not in BASELINE_PI:
        raise ValueError(f"Skenario tidak ditemukan: {scenario_name}")
    if load_condition not in BASELINE_PI[scenario_name]:
        raise ValueError(f"Kondisi beban tidak ditemukan: {load_condition}")

    baseline = BASELINE_PI[scenario_name][load_condition]
    return baseline["PIsc"], baseline["PIv"], baseline["PIpf"]


BASELINE_PISC, BASELINE_PIV, BASELINE_PIPF = get_selected_baseline(
    CURRENT_SCENARIO,
    LOAD_CONDITION,
)


# ============================================================
# BOBOT AHP
# Urutan kriteria:
# PI Total =
# 0.637 * PIsc_norm +
# 0.258 * PIpf_norm +
# 0.105 * PIv_norm
# ============================================================

WEIGHT_SC = 0.637
WEIGHT_V = 0.105
WEIGHT_PF = 0.258


# ============================================================
# PARAMETER PIsc
# ============================================================

DEFAULT_SCC_LIMIT_KA = 40.0
SPECIAL_SCC_LIMIT_KA = {'KRIAN5': 50.0, 'NWARU5': 50.0}


def get_scc_limit_ka(substation_name: str) -> float:
    name = normalize_name(substation_name)
    return SPECIAL_SCC_LIMIT_KA.get(name, DEFAULT_SCC_LIMIT_KA)


# ============================================================
# PARAMETER PIv DAN VIOLATION MONITORING
# ============================================================

V_RATED_PU = 1.0
DV_LIMIT_UNDER_PU = 0.05
DV_LIMIT_OVER_PU = 0.1
VMIN = 0.95
VMAX = 1.1
MAX_LINE_LOADING_PERCENT = 100.0


# ============================================================
# PARAMETER NORMALISASI
# ============================================================

EPSILON = 1e-12
ZERO_BASELINE_PENALTY = 1000000.0


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_DIR = 'D:\Robbyo\S2 UGM\2026\Tesis\Hasil Uji\Final Revisi\SIMULASI GI SAMBIKEREP'
OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    f"{make_safe_filename(CASE_NAME)}_Skenario_3__Beban_Maksimum.xlsx",
)

SAVE_DETAIL_EACH_CONFIG = True


# ============================================================
# BASIC HELPER
# ============================================================

def normalize_name(name: str) -> str:
    """
    Normalisasi nama:
    - uppercase
    - trim spasi depan belakang
    - multiple spaces menjadi single space
    """
    if name is None:
        return ""
    return " ".join(str(name).strip().upper().split())


def compact_name(name: str) -> str:
    """
    Normalisasi agresif untuk matching nama:
    - uppercase
    - hapus spasi dan karakter pemisah umum
    """
    if name is None:
        return ""

    text = str(name).strip().upper()
    for ch in [" ", "-", "_", ".", "/", "\\"]:
        text = text.replace(ch, "")
    return text


def make_safe_filename(text: str) -> str:
    return (
        text.replace(" ", "_")
            .replace("-", "")
            .replace("/", "_")
            .replace("\\", "_")
    )


def log(app: Any, message: str) -> None:
    try:
        app.PrintPlain(message)
    except Exception:
        print(message)


def safe_round(value: Any, ndigits: int = 8) -> Optional[float]:
    if value is None:
        return None
    try:
        number = float(value)
    except Exception:
        return None
    if not math.isfinite(number):
        return None
    return round(number, ndigits)


def as_float(value: Any, default: Optional[float] = None) -> Optional[float]:
    try:
        number = float(value)
    except Exception:
        return default

    if not math.isfinite(number):
        return default

    return number


# ============================================================
# POWERFACTORY APP
# ============================================================

def get_app() -> Any:
    if powerfactory is None:
        raise RuntimeError("powerfactory module not available. Run this script inside DIgSILENT PowerFactory.")

    app = powerfactory.GetApplication()
    if not app:
        raise RuntimeError("Run this script inside DIgSILENT PowerFactory.")

    log(app, "===================================================")
    log(app, f"START MANUAL CONFIG READER: {CASE_NAME}")
    log(app, f"CURRENT_SCENARIO = {CURRENT_SCENARIO}")
    log(app, f"LOAD_CONDITION  = {LOAD_CONDITION}")
    log(app, "Mode            = READ CURRENT ACTIVE CONFIGURATION ONLY")
    log(app, "DS Bus          = NOT changed by script")
    log(app, "BC / Coupler    = NOT opened/closed by script")
    log(app, f"BC Monitor      = {BC_MONITOR_NAME}")
    log(app, f"Baseline PIsc   = {BASELINE_PISC}")
    log(app, f"Baseline PIv    = {BASELINE_PIV}")
    log(app, f"Baseline PIpf   = {BASELINE_PIPF}")
    log(app, "PIpf source     = LINE_LIST only")
    log(app, "SCC Limit       = KRIAN5 & NWARU5 = 50 kA, lainnya = 40 kA")
    log(app, "PI Total        = 0.637*PIsc_norm + 0.258*PIpf_norm + 0.105*PIv_norm")
    log(app, "===================================================")

    return app


def safe_get_attribute(obj: Any, attr: str, default: Any = None) -> Any:
    try:
        value = obj.GetAttribute(attr)
        return default if value is None else value
    except Exception:
        return default


def safe_loc_name(obj: Any, default: str = "") -> str:
    if obj is None:
        return default
    try:
        return obj.loc_name.strip()
    except Exception:
        return default


def get_obj_key(obj: Any) -> Optional[str]:
    if obj is None:
        return None
    try:
        return obj.GetFullName()
    except Exception:
        pass
    try:
        return f"{obj.GetClassName()}::{obj.loc_name.strip()}"
    except Exception:
        return str(obj)


def get_first_attr(obj: Any, attrs: Sequence[str]) -> Any:
    for attr in attrs:
        try:
            val = obj.GetAttribute(attr)
            if val is not None:
                return val
        except Exception:
            pass
    return None


def get_parent_substation_name(obj: Any, max_depth: int = 20) -> Optional[str]:
    current = obj
    for _ in range(max_depth):
        if current is None:
            return None
        try:
            if current.GetClassName() == "ElmSubstat":
                return current.loc_name.strip()
            current = current.GetParent()
        except Exception:
            return None
    return None


def resolve_terminal(side_obj: Any) -> Any:
    if side_obj is None:
        return None
    try:
        if side_obj.GetClassName() == "ElmTerm":
            return side_obj
    except Exception:
        pass
    try:
        terminal = side_obj.cterm
        if terminal is not None:
            return terminal
    except Exception:
        pass
    return None


# ============================================================
# POWERFACTORY COMMAND
# ============================================================

def run_loadflow(app: Any) -> bool:
    app.Rebuild()
    lf = app.GetFromStudyCase("ComLdf")
    if lf is None:
        raise RuntimeError("ComLdf not found in active study case.")

    # Load flow AC balanced, mengikuti template sebelumnya.
    lf.iopt_net = 0
    lf.iopt_o4 = 0

    return lf.Execute() == 0


def prepare_short_circuit(app: Any) -> Any:
    sc = app.GetFromStudyCase("ComShc")
    if sc is None:
        raise RuntimeError("ComShc not found in active study case.")

    # Sesuaikan iopt_mde jika studi Bapak memakai mode yang berbeda.
    # Template sebelumnya memakai iopt_mde = 1.
    sc.iopt_mde = 1
    sc.Rf = 0
    sc.Xf = 0
    sc.iopt_allbus = 0
    sc.iopt_shc = "3psc"
    return sc


# ============================================================
# CACHE BC MONITOR / BUSES / LINES
# ============================================================

def build_elm_coup_lookup(app: Any) -> Dict[str, Dict[str, Any]]:
    objects = list(app.GetCalcRelevantObjects("*.ElmCoup") or [])
    normal = {}
    compact = {}

    for obj in objects:
        name = safe_loc_name(obj)
        normal.setdefault(normalize_name(name), obj)
        compact.setdefault(compact_name(name), obj)

    return {"objects": objects, "normal": normal, "compact": compact}


def find_elm_coup(lookup: Dict[str, Dict[str, Any]], target_name: str) -> Any:
    obj = lookup["normal"].get(normalize_name(target_name))
    if obj is not None:
        return obj
    return lookup["compact"].get(compact_name(target_name))


def cache_bc_monitor(app: Any, bc_name: Optional[str]) -> Tuple[Optional[Dict[str, Any]], List[Dict[str, Any]]]:
    if not bc_name:
        return None, []

    lookup = build_elm_coup_lookup(app)
    obj = find_elm_coup(lookup, bc_name)
    if obj is None:
        return None, [{
            "BC Monitor Target": bc_name,
            "PowerFactory Name": None,
            "Status": "NOT_FOUND",
        }]

    try:
        is_open = bool(obj.IsOpen())
    except Exception:
        is_open = None

    return {
        "target_name": bc_name,
        "pf_name": safe_loc_name(obj),
        "obj": obj,
        "is_open": is_open,
    }, []


def read_bc_monitor(bc_monitor: Optional[Dict[str, Any]]) -> Tuple[Optional[float], Optional[float], Optional[float], Optional[bool], Optional[str]]:
    if not bc_monitor:
        return None, None, None, None, None

    bc_obj = bc_monitor["obj"]
    p_bc = as_float(get_first_attr(bc_obj, BC_P_ATTRS), None)
    q_bc = as_float(get_first_attr(bc_obj, BC_Q_ATTRS), None)

    if p_bc is not None and q_bc is not None:
        s_bc = math.sqrt(p_bc * p_bc + q_bc * q_bc)
    else:
        s_bc = None

    try:
        is_open = bool(bc_obj.IsOpen())
    except Exception:
        is_open = None

    return p_bc, q_bc, s_bc, is_open, bc_monitor.get("pf_name")


def cache_buses(app: Any) -> Tuple[List[Dict[str, Any]], set, set, List[Dict[str, Any]]]:
    substations = {
        safe_loc_name(substation): substation
        for substation in app.GetCalcRelevantObjects("*.ElmSubstat")
    }
    substations_norm = {
        normalize_name(name): substation
        for name, substation in substations.items()
    }

    buses: List[Dict[str, Any]] = []
    missing_rows: List[Dict[str, Any]] = []
    monitored_substation_names = set()
    monitored_bus_keys = set()

    for sub_name in SUBSTAT_LIST:
        substation = substations.get(sub_name)
        if substation is None:
            substation = substations_norm.get(normalize_name(sub_name))

        if substation is None:
            missing_rows.append({"Substation": sub_name, "Status": "NOT_FOUND"})
            log(app, f"Warning: Substation not found: {sub_name}")
            continue

        actual_sub_name = safe_loc_name(substation) or sub_name
        monitored_substation_names.add(actual_sub_name)

        all_terms = list(substation.GetContents("*.ElmTerm", 1) or [])

        for bus in all_terms:
            usage = safe_get_attribute(bus, "iUsage")
            try:
                if int(usage) != 0:
                    continue
            except Exception:
                continue

            key = get_obj_key(bus)
            if not key or key in monitored_bus_keys:
                continue

            monitored_bus_keys.add(key)
            buses.append({
                "substation": actual_sub_name,
                "bus_name": safe_loc_name(bus),
                "obj": bus,
                "key": key,
            })

    log(app, f"Total monitored buses from SUBSTAT_LIST = {len(buses)}")
    return buses, monitored_bus_keys, monitored_substation_names, missing_rows


def get_line_terminals(line_obj: Any) -> Tuple[Any, Any]:
    cub1 = None
    cub2 = None

    try:
        cub1 = line_obj.bus1
    except Exception:
        pass
    try:
        cub2 = line_obj.bus2
    except Exception:
        pass

    if cub1 is None:
        cub1 = safe_get_attribute(line_obj, "bus1")
    if cub2 is None:
        cub2 = safe_get_attribute(line_obj, "bus2")

    return resolve_terminal(cub1), resolve_terminal(cub2)


def cache_lines(app: Any) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    lines: List[Dict[str, Any]] = []
    missing_rows: List[Dict[str, Any]] = []
    seen_line_keys = set()

    target_line_map_normal = {normalize_name(line_name): line_name for line_name in LINE_LIST}
    target_line_map_compact = {compact_name(line_name): line_name for line_name in LINE_LIST}

    found_target_names = set()

    for line in app.GetCalcRelevantObjects("*.ElmLne"):
        line_name_pf = safe_loc_name(line)
        line_key_normal = normalize_name(line_name_pf)
        line_key_compact = compact_name(line_name_pf)

        target_name = None
        if line_key_normal in target_line_map_normal:
            target_name = target_line_map_normal[line_key_normal]
        elif line_key_compact in target_line_map_compact:
            target_name = target_line_map_compact[line_key_compact]

        if target_name is None:
            continue

        outserv = safe_get_attribute(line, "outserv", 0)
        try:
            if int(outserv) == 1:
                missing_rows.append({
                    "Target Line Name": target_name,
                    "PowerFactory Name": line_name_pf,
                    "Status": "FOUND_BUT_OUT_OF_SERVICE",
                })
                found_target_names.add(target_name)
                continue
        except Exception:
            pass

        line_obj_key = get_obj_key(line)
        if not line_obj_key or line_obj_key in seen_line_keys:
            continue

        seen_line_keys.add(line_obj_key)
        found_target_names.add(target_name)

        term1, term2 = get_line_terminals(line)
        term1_sub = get_parent_substation_name(term1) or ""
        term2_sub = get_parent_substation_name(term2) or ""

        lines.append({
            "line_name": line_name_pf,
            "line_name_target": target_name,
            "selected_side": "max_current_side",
            "obj": line,
            "bus_name": f"{safe_loc_name(term1)} / {safe_loc_name(term2)}",
            "substation": f"{term1_sub} / {term2_sub}",
        })

    for original_name in LINE_LIST:
        if original_name not in found_target_names:
            missing_rows.append({
                "Target Line Name": original_name,
                "PowerFactory Name": None,
                "Status": "NOT_FOUND",
            })

    log(app, f"Total monitored lines from LINE_LIST = {len(lines)}")
    log(app, f"Total missing/out-service lines from LINE_LIST = {len(missing_rows)}")

    if missing_rows:
        log(app, "Warning: beberapa line di LINE_LIST tidak ditemukan atau out of service:")
        for row in missing_rows:
            log(app, f" - {row['Target Line Name']} | {row['Status']}")

    return lines, missing_rows


# ============================================================
# PERFORMANCE INDEX - PIpf
# ============================================================

def calculate_pipf_total(lines: Sequence[Dict[str, Any]], save_details: bool = False) -> Tuple[float, List[Dict[str, Any]], List[str]]:
    total_pipf = 0.0
    details: List[Dict[str, Any]] = []
    violated_items: List[str] = []

    for item in lines:
        line_obj = item["obj"]

        loading = as_float(safe_get_attribute(line_obj, "m:loading", 0.0), 0.0)
        i_bus1 = as_float(safe_get_attribute(line_obj, "m:I1:bus1", 0.0), 0.0)
        i_bus2 = as_float(safe_get_attribute(line_obj, "m:I1:bus2", 0.0), 0.0)

        i_bus1 = abs(i_bus1 or 0.0)
        i_bus2 = abs(i_bus2 or 0.0)
        i_actual = max(i_bus1, i_bus2)

        i_limit = as_float(safe_get_attribute(line_obj, "c:curnom", 0.0), 0.0)
        i_limit = i_limit or 0.0

        if i_limit <= 0.0:
            pipf_i = 0.0
        else:
            pipf_i = 0.5 * ((i_actual / i_limit) ** 2)

        total_pipf += pipf_i

        line_label = item.get("line_name_target") or item.get("line_name")
        is_violated = loading is not None and loading > MAX_LINE_LOADING_PERCENT
        if is_violated:
            violated_items.append(line_label)

        if save_details:
            details.append({
                "Substation": item["substation"],
                "Bus": item["bus_name"],
                "Line": item["line_name"],
                "Target Line Name": item.get("line_name_target"),
                "Selected Side": "Max(bus1,bus2)",
                "Loading (%)": safe_round(loading, 6),
                "Loading Limit (%)": MAX_LINE_LOADING_PERCENT,
                "Loading Violated": is_violated,
                "I bus1 (kA)": safe_round(i_bus1, 6),
                "I bus2 (kA)": safe_round(i_bus2, 6),
                "I actual max (kA)": safe_round(i_actual, 6),
                "I limit (kA)": safe_round(i_limit, 6),
                "PIpf_i = 0.5*(Iactual/Ilimit)^2": safe_round(pipf_i, 8),
            })

    return total_pipf, details, sorted(set(violated_items))


# ============================================================
# PERFORMANCE INDEX - PIsc
# ============================================================

def calculate_pisc_total(sc: Any, buses: Sequence[Dict[str, Any]], save_details: bool = False) -> Tuple[Optional[float], List[Dict[str, Any]], List[str], str]:
    total_pisc = 0.0
    details: List[Dict[str, Any]] = []
    violated_subs: List[str] = []

    for item in buses:
        substation_name = item["substation"]
        bus = item["obj"]
        sc.shcobj = bus

        if sc.Execute() != 0:
            return None, details, violated_subs, f"SC_EXECUTE_FAILED: {item['substation']} / {item['bus_name']}"

        isc = as_float(safe_get_attribute(bus, "m:Ikss", None), None)
        if isc is None:
            isc = as_float(safe_get_attribute(bus, "m:ikss", None), None)

        if isc is None:
            return None, details, violated_subs, f"SC_RESULT_MISSING: {item['substation']} / {item['bus_name']}"

        scc_limit_ka = get_scc_limit_ka(substation_name)

        if isc > scc_limit_ka:
            pisc_i = (isc / scc_limit_ka) ** 2
            violation = True
            violated_subs.append(substation_name)
        else:
            pisc_i = 0.0
            violation = False

        total_pisc += pisc_i

        if save_details:
            details.append({
                "Substation": item["substation"],
                "Bus": item["bus_name"],
                "Isc / Ikss (kA)": safe_round(isc, 6),
                "SCC Limit Used (kA)": safe_round(scc_limit_ka, 6),
                "PIsc_i = (Isc/SCC Limit)^2 if Isc>Limit": safe_round(pisc_i, 8),
                "Violation": violation,
            })

    return total_pisc, details, sorted(set(violated_subs)), "OK"


# ============================================================
# PERFORMANCE INDEX - PIv
# ============================================================

def calculate_piv_total(buses: Sequence[Dict[str, Any]], save_details: bool = False) -> Tuple[Optional[float], List[Dict[str, Any]], List[str], str]:
    total_piv = 0.0
    details: List[Dict[str, Any]] = []
    violated_subs: List[str] = []

    for item in buses:
        sub_name = item["substation"]
        bus_name = item["bus_name"]
        bus_obj = item["obj"]

        vpu = as_float(safe_get_attribute(bus_obj, "m:u", None), None)
        vkv = as_float(safe_get_attribute(bus_obj, "m:Ul", None), None)

        if vpu is None:
            return None, details, violated_subs, f"VOLTAGE_RESULT_MISSING: {sub_name} / {bus_name}"

        deviation = abs(vpu - V_RATED_PU)

        if vpu < V_RATED_PU:
            dv_limit = DV_LIMIT_UNDER_PU
        elif vpu > V_RATED_PU:
            dv_limit = DV_LIMIT_OVER_PU
        else:
            dv_limit = DV_LIMIT_UNDER_PU

        piv_i = 0.5 * ((deviation / dv_limit) ** 2)
        total_piv += piv_i

        if vpu < VMIN:
            status = "LOW"
            violated_subs.append(sub_name)
        elif vpu > VMAX:
            status = "HIGH"
            violated_subs.append(sub_name)
        else:
            status = "NORMAL"

        if save_details:
            details.append({
                "Substation": sub_name,
                "Bus": bus_name,
                "Vpu": safe_round(vpu, 6),
                "VkV": safe_round(vkv, 3) if vkv is not None else None,
                "Vrated_pu": V_RATED_PU,
                "DeltaVlim_pu": dv_limit,
                "Deviation": safe_round(deviation, 8),
                "PIv_i = 0.5*(Deviation/DeltaVlim)^2": safe_round(piv_i, 8),
                "Status": status,
            })

    return total_piv, details, sorted(set(violated_subs)), "OK"


# ============================================================
# NORMALISASI DAN PI TOTAL
# ============================================================

def normalize_by_baseline(value: Optional[float], baseline: float) -> Optional[float]:
    if value is None:
        return None
    if abs(baseline) <= EPSILON:
        if abs(value) <= EPSILON:
            return 1.0
        return ZERO_BASELINE_PENALTY
    return value / baseline


def calculate_pi_total(pisc_total: float, piv_total: float, pipf_total: float) -> Tuple[
    Optional[float], Optional[float], Optional[float], Optional[float], Optional[float], Optional[float], Optional[float]
]:
    pisc_norm = normalize_by_baseline(pisc_total, BASELINE_PISC)
    piv_norm = normalize_by_baseline(piv_total, BASELINE_PIV)
    pipf_norm = normalize_by_baseline(pipf_total, BASELINE_PIPF)

    if pisc_norm is None or piv_norm is None or pipf_norm is None:
        return None, pisc_norm, piv_norm, pipf_norm, None, None, None

    pisc_contribution = WEIGHT_SC * pisc_norm
    piv_contribution = WEIGHT_V * piv_norm
    pipf_contribution = WEIGHT_PF * pipf_norm
    pi_total = pisc_contribution + piv_contribution + pipf_contribution

    return pi_total, pisc_norm, piv_norm, pipf_norm, pisc_contribution, piv_contribution, pipf_contribution


# ============================================================
# MAIN PROGRAM
# ============================================================

def run_manual_configuration_reader() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    app = get_app()
    t0 = time.perf_counter()

    bc_monitor, missing_bc_rows = cache_bc_monitor(app, BC_MONITOR_NAME)
    buses, monitored_bus_keys, monitored_substation_names, missing_sub_rows = cache_buses(app)
    lines, missing_line_rows = cache_lines(app)

    if not buses:
        raise RuntimeError("No buses found from SUBSTAT_LIST.")
    if not lines:
        raise RuntimeError("No lines found from LINE_LIST.")

    log(app, "Running load flow for current manual configuration...")
    if not run_loadflow(app):
        raise RuntimeError("Load flow failed. Cek konfigurasi manual di PowerFactory.")

    p_bc, q_bc, s_bc, bc_is_open, bc_pf_name = read_bc_monitor(bc_monitor)

    pipf_total, line_details, viol_pf = calculate_pipf_total(
        lines,
        save_details=SAVE_DETAIL_EACH_CONFIG,
    )

    piv_total, v_details, viol_v, piv_status = calculate_piv_total(
        buses,
        save_details=SAVE_DETAIL_EACH_CONFIG,
    )
    if piv_total is None:
        raise RuntimeError(piv_status)

    log(app, "Running short circuit calculation for current manual configuration...")
    sc = prepare_short_circuit(app)
    pisc_total, sc_details, viol_sc, pisc_status = calculate_pisc_total(
        sc,
        buses,
        save_details=SAVE_DETAIL_EACH_CONFIG,
    )
    if pisc_total is None:
        raise RuntimeError(pisc_status)

    (
        pi_total,
        pisc_norm,
        piv_norm,
        pipf_norm,
        pisc_contribution,
        piv_contribution,
        pipf_contribution,
    ) = calculate_pi_total(
        pisc_total=pisc_total,
        piv_total=piv_total,
        pipf_total=pipf_total,
    )

    violated = len(set(viol_sc + viol_v + viol_pf)) > 0
    elapsed = time.perf_counter() - t0

    summary_row = {
        "Case": CASE_NAME,
        "Current Scenario": CURRENT_SCENARIO,
        "Load Condition": LOAD_CONDITION,
        "Mode": "Manual configuration already set in PowerFactory",
        "Script Action DS Bus": "NO_CHANGE",
        "Script Action BC / Coupler": "NO_CHANGE",

        "PI Total": safe_round(pi_total, 8),

        "PIsc Total": safe_round(pisc_total, 8),
        "PIsc Baseline": BASELINE_PISC,
        f"PIsc Norm = PIsc Total / {BASELINE_PISC}": safe_round(pisc_norm, 8),
        "Weight PIsc": WEIGHT_SC,
        f"PIsc Contribution = {WEIGHT_SC}*PIsc Norm": safe_round(pisc_contribution, 8),

        "PIv Total": safe_round(piv_total, 8),
        "PIv Baseline": BASELINE_PIV,
        f"PIv Norm = PIv Total / {BASELINE_PIV}": safe_round(piv_norm, 8),
        "Weight PIv": WEIGHT_V,
        f"PIv Contribution = {WEIGHT_V}*PIv Norm": safe_round(piv_contribution, 8),

        "PIpf Total": safe_round(pipf_total, 8),
        "PIpf Baseline": BASELINE_PIPF,
        f"PIpf Norm = PIpf Total / {BASELINE_PIPF}": safe_round(pipf_norm, 8),
        "Weight PIpf": WEIGHT_PF,
        f"PIpf Contribution = {WEIGHT_PF}*PIpf Norm": safe_round(pipf_contribution, 8),

        "P Monitor BC (MW)": safe_round(p_bc, 6),
        "Q Monitor BC (Mvar)": safe_round(q_bc, 6),
        "S Monitor BC (MVA)": safe_round(s_bc, 6),
        "BC Monitor Target Name": BC_MONITOR_NAME,
        "BC Monitor PF Name": bc_pf_name,
        "BC Monitor Is Open": bc_is_open,

        "Violated": violated,
        "Viol SC": ", ".join(viol_sc),
        "Viol V": ", ".join(viol_v),
        "Viol PF": ", ".join(viol_pf),

        "Total Monitored Buses": len(buses),
        "Total PIpf Lines Found": len(lines),
        "Total Missing/Out-service Lines": len(missing_line_rows),
        "Execution Time (s)": safe_round(elapsed, 2),
        "Status": "OK",
    }

    df_summary = pd.DataFrame([summary_row])

    detail_sc_rows = []
    detail_v_rows = []
    detail_line_rows = []

    for row in sc_details:
        detail_sc_rows.append({
            "Case": CASE_NAME,
            "PIsc Total": safe_round(pisc_total, 8),
            "PIsc Norm": safe_round(pisc_norm, 8),
            "PIsc Contribution": safe_round(pisc_contribution, 8),
            **row,
        })

    for row in v_details:
        detail_v_rows.append({
            "Case": CASE_NAME,
            "PIv Total": safe_round(piv_total, 8),
            "PIv Norm": safe_round(piv_norm, 8),
            "PIv Contribution": safe_round(piv_contribution, 8),
            **row,
        })

    for row in line_details:
        detail_line_rows.append({
            "Case": CASE_NAME,
            "PIpf Total": safe_round(pipf_total, 8),
            "PIpf Norm": safe_round(pipf_norm, 8),
            "PIpf Contribution": safe_round(pipf_contribution, 8),
            **row,
        })

    df_detail_sc = pd.DataFrame(detail_sc_rows)
    df_detail_v = pd.DataFrame(detail_v_rows)
    df_detail_line = pd.DataFrame(detail_line_rows)

    df_missing_bc = pd.DataFrame(missing_bc_rows)
    df_missing_sub = pd.DataFrame(missing_sub_rows)
    df_missing_line = pd.DataFrame(missing_line_rows)

    df_param = pd.DataFrame([
        {"Parameter": "CURRENT_SCENARIO", "Value": CURRENT_SCENARIO},
        {"Parameter": "LOAD_CONDITION", "Value": LOAD_CONDITION},
        {"Parameter": "MODE", "Value": "READ_ONLY_MANUAL_CONFIGURATION"},
        {"Parameter": "BC_MONITOR_NAME", "Value": BC_MONITOR_NAME},
        {"Parameter": "BASELINE_PISC", "Value": BASELINE_PISC},
        {"Parameter": "BASELINE_PIV", "Value": BASELINE_PIV},
        {"Parameter": "BASELINE_PIPF", "Value": BASELINE_PIPF},
        {"Parameter": "WEIGHT_SC", "Value": WEIGHT_SC},
        {"Parameter": "WEIGHT_V", "Value": WEIGHT_V},
        {"Parameter": "WEIGHT_PF", "Value": WEIGHT_PF},
        {"Parameter": "DEFAULT_SCC_LIMIT_KA", "Value": DEFAULT_SCC_LIMIT_KA},
        {"Parameter": "SPECIAL_SCC_LIMIT_KA", "Value": str(SPECIAL_SCC_LIMIT_KA)},
        {"Parameter": "V_RATED_PU", "Value": V_RATED_PU},
        {"Parameter": "DV_LIMIT_UNDER_PU", "Value": DV_LIMIT_UNDER_PU},
        {"Parameter": "DV_LIMIT_OVER_PU", "Value": DV_LIMIT_OVER_PU},
        {"Parameter": "VMIN", "Value": VMIN},
        {"Parameter": "VMAX", "Value": VMAX},
        {"Parameter": "MAX_LINE_LOADING_PERCENT", "Value": MAX_LINE_LOADING_PERCENT},
        {"Parameter": "OUTPUT_FILE", "Value": OUTPUT_FILE},
        {"Parameter": "NOTE", "Value": "Script tidak mengubah DS Bus dan tidak membuka/menutup BC."},
    ])

    with pd.ExcelWriter(OUTPUT_FILE) as writer:
        df_summary.to_excel(writer, sheet_name="Summary_Manual_Config", index=False)

        if not df_detail_sc.empty:
            df_detail_sc.to_excel(writer, sheet_name="Detail_PIsc", index=False)
        if not df_detail_v.empty:
            df_detail_v.to_excel(writer, sheet_name="Detail_PIv", index=False)
        if not df_detail_line.empty:
            df_detail_line.to_excel(writer, sheet_name="Detail_PIpf_LINE_LIST", index=False)

        if not df_missing_bc.empty:
            df_missing_bc.to_excel(writer, sheet_name="Missing_BC_Monitor", index=False)
        if not df_missing_sub.empty:
            df_missing_sub.to_excel(writer, sheet_name="Missing_Substation", index=False)
        if not df_missing_line.empty:
            df_missing_line.to_excel(writer, sheet_name="Missing_LINE_LIST", index=False)

        df_param.to_excel(writer, sheet_name="Parameters", index=False)

    log(app, "===================================================")
    log(app, "FINISHED MANUAL CONFIG READER")
    log(app, f"Saved Excel : {OUTPUT_FILE}")
    log(app, f"Execution   : {elapsed:.2f} s")
    log(app, f"PI Total    : {pi_total:.8f}" if pi_total is not None else "PI Total    : None")
    log(app, f"PIsc Total  : {pisc_total:.8f}")
    log(app, f"PIv Total   : {piv_total:.8f}")
    log(app, f"PIpf Total  : {pipf_total:.8f}")
    log(app, f"Violated    : {violated}")
    log(app, "Catatan     : DS dan BC tidak diubah oleh script.")
    log(app, "===================================================")


if __name__ == "__main__":
    run_manual_configuration_reader()


Saved Excel : D:\Robbyo\S2 UGM6\Tesis\Hasil Uji\Final Revisi\SIMULASI GI SAMBIKEREP\Skenario_3__Beban_Maksimum_Skenario_3__Beban_Maksimum.xlsx


In [11]:
import powerfactory
# ==========================
# TARGET LOAD DATA - BEBAN RATA-RATA
# ==========================
LOAD_TARGETS = [
    ("4KLANG5_TD1", 20.660000, 6.030000),
    ("4KLANG5_TD2", 35.060000, 8.290000),
    ("4SWHAN5_TD1", 20.620000, 6.750000),
    ("4SWHAN5_TD3", 20.490000, 6.680000),
    ("4DARMO5_TD1", 28.490000, 8.560000),
    ("4DARMO5_TD2", 26.010000, 5.990000),
    ("4DARMO5_TD3", 18.260000, 3.950000),
    ("4ALTAP5_TD1", 14.320000, 4.720000),
    ("4ALTAP5_TD2", 11.840000, 2.460000),
    ("4ALTAP5_TD3", 6.040000, 1.570000),
    ("4SMKRP5_TD1", 17.390000, 5.080000),
    ("4SMKRP5_TD2", 13.880000, 0.620000),
    ("4WLMAR5_KTT", 38.170000, 24.640000),
    ("4BAMBE5_TD1", 26.950000, 9.820000),
    ("4BAMBE5_TD2", 20.540000, 9.030000),
    ("4GNSRI5_TD1", 6.200000, 1.840000),
    ("4GNSRI5_TD2", 3.930000, 0.840000),
    ("4PKMIA5_TD1", 3.900000, 1.580000),
    ("4PKMIA5_TD2", 22.020000, 8.090000),
    ("4PKMIA5_TD3", 17.840000, 7.910000),
    ("4WARU5_TD3", 21.640000, 6.840000),
    ("4WARU5_TD4", 20.750000, 6.870000),
    ("4WARU5_TD5", 14.580000, 5.410000),
    ("4WARU5_TD6", 18.600000, 5.470000),
    ("4WARU5_TD7", 21.600000, 7.180000),
    ("4RNKUT5_TD1", 31.190000, 8.800000),
    ("4RNKUT5_TD2", 35.350000, 9.930000),
    ("4RNKUT5_TD3", 16.060000, 5.920000),
    ("4RNKUT5_TD4", 28.700000, 7.990000),
    ("4RNKUT5_TD5", 30.940000, 11.050000),
    ("4BDRAN5_TD2", 25.090000, 9.500000),
    ("4BDRAN5_TD3", 19.400000, 6.540000),
    ("4BDRAN5_TD4", 28.290000, 10.180000),
    ("4BDRAN5_TD6", 35.340000, 11.460000),
    ("4BDRAN4_TD5", 10.660000, 3.390000),
    ("4SLILO5_TD1", 16.920000, 3.840000),
    ("4SLILO5_TD2", 24.800000, 6.500000),
    ("4SLILO5_TD3", 13.770000, 3.080000),
    ("4SLILO5_TD4", 23.610000, 5.980000),
    ("4HANIL5_KTT", 28.220000, 17.390000),
    ("4PTJTS5_KTT", 7.950000, 4.600000),
    ("4KLSRI5_TD1", 23.510000, 6.020000),
    ("4WKRMO5_TD1", 24.080000, 6.730000),
    ("4WKRMO5_TD2", 22.140000, 6.020000),
    ("4WKRMO5_TD3", 8.170000, 1.540000),
    ("4NGAGL5_TD1", 4.470000, 1.230000),
    ("4NGAGL5_TD2", 22.270000, 5.830000),
    ("4KJRAN5_TD1", 19.970000, 4.890000),
    ("4KJRAN5_TD2", 21.800000, 4.890000),
    ("4KJRAN5_TD3", 21.090000, 5.410000),
    ("4SDRJO5_TD1", 44.490000, 12.680000),
    ("4SDRJO5_TD2", 37.940000, 13.090000),
    ("4SDATI5_TD1", 11.360000, 3.560000),
    ("4SDATI5_TD2", 19.690000, 6.330000),
    ("4SIMPG5_TD1", 18.120000, 5.040000),
    ("4SIMPG5_TD2", 10.280000, 2.830000),
    ("4GLTMR5_TD1", 5.260000, 1.790000),
    ("4GLTMR5_TD2", 7.060000, 2.370000),
    ("4KDING5_TD1", 12.330000, 4.040000),
    ("4BKLAN5_TD1", 34.320000, 8.870000),
    ("4BKLAN5_TD2", 19.700000, 8.100000),
    ("4TNDES5_TD1", 18.310000, 4.436000),
    ("4TNDES5_TD2", 18.060000, 5.234000),
    ("4TNDES5_TD3", 5.290000, 3.796000),
    ("4TNDES5_TD4", 2.420000, 0.873000),
    ("4TNDES5_TD5", 24.890000, 6.977000),
    ("4PERAK5_TD1", 9.050000, 4.904000),
    ("4UDAAN5_TD1", 14.640000, 3.998000),
    ("4UDAAN5_TD2", 8.870000, 1.986000),
    ("4KPANG5_TD1", 22.500000, 6.164000),
    ("4KPANG5_TD2", 10.180000, 3.048000),
    ("4KRBAN5_TD1", 13.440000, 2.856000),
    ("4KRBAN5_TD2", 13.160000, 3.738000),
    ("4KRBAN5_TD3", 8.640000, 2.104000),
    ("4UJUNG5_TD1", 4.250000, 2.160000),
    ("4UJUNG5_TD2", 12.120000, 2.525000),
    ("4INDSM5_KTT1", 12.510000, 7.087000),
    ("4SAMPG5_TD1", 23.790000, 4.929000),
    ("4SAMPG5_TD2", 36.980000, 9.848000),
    ("4PMKSN5_TD1", 27.210000, 5.578000),
    ("4PMKSN5_TD2", 36.390000, 8.701000),
    ("4GULUK5_TD1", 21.330000, 4.507000),
    ("4SMNEP5_TD1", 23.370000, 4.826000),
    ("4SMNEP5_TD2", 27.180000, 5.256000),
]


def get_all_loads(app):
    all_loads = []
    all_loads.extend(app.GetCalcRelevantObjects("*.ElmLod"))
    all_loads.extend(app.GetCalcRelevantObjects("*.ElmLodmv"))
    all_loads.extend(app.GetCalcRelevantObjects("*.ElmLodlv"))
    return all_loads


def build_load_map(app):
    load_map = {}
    for lod in get_all_loads(app):
        name = lod.loc_name.strip()
        load_map.setdefault(name, []).append(lod)
    return load_map


def run_loadflow(app):
    lf = app.GetFromStudyCase("ComLdf")
    if lf is None:
        print("Load flow object not found")
        return False

    rc = lf.Execute()
    if rc != 0:
        print("Load flow failed")
        return False

    print("\nLoad Flow Completed")
    return True


def update_average_loads():
    app = powerfactory.GetApplication()

    if app is None:
        print("Cannot connect to PowerFactory")
        return

    load_map = build_load_map(app)

    # Group target data by load name while preserving row order from Excel
    grouped_targets = {}
    for load_name, p_mw, q_mvar in LOAD_TARGETS:
        grouped_targets.setdefault(load_name, []).append((p_mw, q_mvar))

    total_updated = 0
    not_found = []
    mismatch = []

    print("=== UPDATE LOAD P/Q BY LOAD NAME (BEBAN RATA-RATA) ===\n")

    for load_name, target_list in grouped_targets.items():
        matched_loads = load_map.get(load_name, [])

        if not matched_loads:
            for _ in target_list:
                not_found.append(load_name)
            print(f"[NOT FOUND] {load_name} -> target count = {len(target_list)}")
            continue

        if len(matched_loads) != len(target_list):
            mismatch.append((load_name, len(target_list), len(matched_loads)))
            print(
                f"[COUNT MISMATCH] {load_name} -> "
                f"target count = {len(target_list)}, found in model = {len(matched_loads)}"
            )

        # Apply sequentially up to the minimum count
        n_apply = min(len(target_list), len(matched_loads))

        for idx in range(n_apply):
            new_p_mw, new_q_mvar = target_list[idx]
            target = matched_loads[idx]

            old_p = target.GetAttribute("plini")
            old_q = target.GetAttribute("qlini")

            target.SetAttribute("plini", new_p_mw)
            target.SetAttribute("qlini", new_q_mvar)

            print(
                f"{load_name:15}#{idx+1} "
                f"P: {old_p:10.4f} -> {new_p_mw:10.4f} MW   "
                f"Q: {old_q:10.4f} -> {new_q_mvar:10.4f} Mvar"
            )
            total_updated += 1

    print("\n=== SUMMARY ===")
    print(f"Total updated : {total_updated}")
    print(f"Not found     : {len(not_found)}")
    print(f"Mismatch      : {len(mismatch)}")

    if not_found:
        print("\nLoad names not found:")
        for name in sorted(set(not_found)):
            print(f"- {name}")

    if mismatch:
        print("\nLoad count mismatches:")
        for name, target_cnt, found_cnt in mismatch:
            print(f"- {name}: target={target_cnt}, found={found_cnt}")

    run_loadflow(app)


if __name__ == "__main__":
    update_average_loads()


=== UPDATE LOAD P/Q BY LOAD NAME (BEBAN RATA-RATA) ===

4KLANG5_TD1    #1 P:    20.6600 ->    20.6600 MW   Q:     6.0300 ->     6.0300 Mvar
4KLANG5_TD2    #1 P:    35.0600 ->    35.0600 MW   Q:     8.2900 ->     8.2900 Mvar
4SWHAN5_TD1    #1 P:    20.6200 ->    20.6200 MW   Q:     6.7500 ->     6.7500 Mvar
4SWHAN5_TD3    #1 P:    20.4900 ->    20.4900 MW   Q:     6.6800 ->     6.6800 Mvar
4DARMO5_TD1    #1 P:    28.4900 ->    28.4900 MW   Q:     8.5600 ->     8.5600 Mvar
4DARMO5_TD2    #1 P:    26.0100 ->    26.0100 MW   Q:     5.9900 ->     5.9900 Mvar
4DARMO5_TD3    #1 P:    18.2600 ->    18.2600 MW   Q:     3.9500 ->     3.9500 Mvar
4ALTAP5_TD1    #1 P:    14.3200 ->    14.3200 MW   Q:     4.7200 ->     4.7200 Mvar
4ALTAP5_TD2    #1 P:    11.8400 ->    11.8400 MW   Q:     2.4600 ->     2.4600 Mvar
4ALTAP5_TD3    #1 P:     6.0400 ->     6.0400 MW   Q:     1.5700 ->     1.5700 Mvar
4SMKRP5_TD1    #1 P:    17.3900 ->    17.3900 MW   Q:     5.0800 ->     5.0800 Mvar
4SMKRP5_TD2    #1 P: